In [85]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [86]:
df = pd.read_csv("data (2).csv")

In [87]:
df

,patient_identifier,text,has_cancer,has_diabetes,test_set
0,2200,DISCHARGE SUMMARY:\n\nPatient Name: [Redacted]...,1.0,0.0,0
1,645,Discharge Summary:\n\nPatient: [Name]\n\nMedic...,0.0,0.0,0
2,2563,Discharge Summary:\nPatient name: [REDACTED]\n...,0.0,0.0,0
3,2275,Discharge Summary:\n\nPatient: 59-year-old Ita...,1.0,0.0,0
4,1828,Hospital Course:\n\nThe 80-year-old male prese...,0.0,1.0,0
...,...,...,...,...,...
1995,699,Discharge Summary:\n\nPatient: 50-year-old mal...,NaN,NaN,0
1996,2334,Discharge Summary:\n\nPatient Name: Unavailabl...,NaN,NaN,0
1997,1868,Discharge Summary:\n\nPatient was a 12-year-ol...,NaN,NaN,0
1998,449,Discharge Summary:\n\nPatient Name: [redacted]...,NaN,NaN,0


In [88]:
labeled_data = df[df['test_set']==0].dropna()

In [89]:
unlabeled = df[(df['has_cancer'].isna()) & (df['has_diabetes'].isna())]

In [90]:
final_test_df = df[df['test_set'] == 1]

In [91]:
final_test_df

,patient_identifier,text,has_cancer,has_diabetes,test_set
65,2079,Discharge summary:\n\nPatient Name: [REDACTED]...,NaN,NaN,1
79,1385,DISCHARGE SUMMARY:\n\nPatient Name: Not Provid...,NaN,NaN,1
81,262,Hospital Course:\n\nAdmitted with: red and pai...,NaN,NaN,1
86,3000,Discharge Summary:\n\nPatient Information:\n- ...,NaN,NaN,1
89,2177,Discharge Summary:\n\nPatient Identification:\...,NaN,NaN,1
...,...,...,...,...,...
1902,989,Discharge Summary:\n\nPatient Name: XXX\nGende...,NaN,NaN,1
1939,1613,Discharge Summary:\n\nPatient Name: [Name]\nDa...,NaN,NaN,1
1946,2896,"Discharge Summary:\n\nPatient: Female, 57 year...",NaN,NaN,1
1948,2406,Hospital Course:\n\nA 61-year-old male patient...,NaN,NaN,1


## EDA

In [92]:
def analyze_class_distribution(df, title_suffix=''):
    """
    Analyze and visualize class distribution for cancer and diabetes labels.
    
    Args:
        df: DataFrame with 'has_cancer' and 'has_diabetes' columns
        title_suffix: Optional suffix for plot title
    
    Returns:
        df with added 'combined_label' column
    """
    # Create combined label column
    def get_combined_label(row):
        has_cancer = row['has_cancer'] == 1
        has_diabetes = row['has_diabetes'] == 1
        
        if has_cancer and has_diabetes:
            return 'Both'
        elif has_cancer:
            return 'Cancer Only'
        elif has_diabetes:
            return 'Diabetes Only'
        else:
            return 'Neither'
    
    df['combined_label'] = df.apply(get_combined_label, axis=1)
    
    # Class distribution table
    class_counts = df['combined_label'].value_counts()
    class_pcts = df['combined_label'].value_counts(normalize=True) * 100
    
    class_summary = pd.DataFrame({
        'Count': class_counts,
        'Percentage': class_pcts.round(2)
    })
    
    print(f"Combined Class Distribution{title_suffix}:")
    print(class_summary)
    print(f"\nTotal records: {len(df)}")
    
    # Interactive plotly visualization
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=class_counts.index,
        y=class_counts.values,
        text=[f"{count}<br>({pct:.1f}%)" for count, pct in zip(class_counts.values, class_pcts.values)],
        textposition='auto',
        marker_color=['#e74c3c', '#f39c12', '#3498db', '#95a5a6']
    ))
    
    fig.update_layout(
        title=f'Class Distribution: Cancer, Diabetes, Both, Neither{title_suffix}',
        xaxis_title='Category',
        yaxis_title='Count',
        height=500
    )
    
    fig.show()
    
    return df


In [93]:
analyze_class_distribution(labeled_data)

Combined Class Distribution:
                Count  Percentage
combined_label                   
Neither            27        54.0
Cancer Only        18        36.0
Diabetes Only       3         6.0
Both                2         4.0

Total records: 50


,patient_identifier,text,has_cancer,has_diabetes,test_set,combined_label
0,2200,DISCHARGE SUMMARY:\n\nPatient Name: [Redacted]...,1.0,0.0,0,Cancer Only
1,645,Discharge Summary:\n\nPatient: [Name]\n\nMedic...,0.0,0.0,0,Neither
2,2563,Discharge Summary:\nPatient name: [REDACTED]\n...,0.0,0.0,0,Neither
3,2275,Discharge Summary:\n\nPatient: 59-year-old Ita...,1.0,0.0,0,Cancer Only
4,1828,Hospital Course:\n\nThe 80-year-old male prese...,0.0,1.0,0,Diabetes Only
5,1523,Discharge Summary:\n\nPatient Name: [Redacted]...,1.0,0.0,0,Cancer Only
6,157,[Hospital Course]\n\nThe 60-year-old Japanese ...,0.0,0.0,0,Neither
7,3133,Discharge Summary:\n\nPatient Information:\nTh...,0.0,0.0,0,Neither
8,1949,Discharge Summary:\n\nPatient Name: Japanese m...,1.0,0.0,0,Cancer Only
9,498,Discharge Summary\n\nPatient Name: [Redacted]\...,0.0,0.0,0,Neither


In [94]:
f"Patience that are not labeled: {len(unlabeled)}"

'Patience that are not labeled: 1950'

In [95]:
def parse_patient_text(text):
    """
    Parse patient description text into structured columns.
    Extracts headers (text before ':') as column names and values after ':'.
    
    Args:
        text: Patient description string with headers denoted by ':'
    
    Returns:
        Dictionary with parsed key-value pairs
    """
    if pd.isna(text):
        return {}
    
    parsed = {}
    lines = text.split('\n')
    for line in lines:
        if ':' in line:
            # Split on first colon only
            parts = line.split(':', 1)
            if len(parts) == 2:
                key = parts[0].strip()
                value = parts[1].strip()
                if key and value:  # Only add if both key and value exist
                    parsed[key] = value
    
    return parsed


In [96]:
text_col = 'text' 

# Parse all records
parsed_data = labeled_data[text_col].apply(parse_patient_text)

# Convert list of dicts to DataFrame
parsed_df = pd.DataFrame(parsed_data.tolist())

# Combine with original df
labeled_data_expanded = pd.concat([labeled_data, parsed_df], axis=1)

print(f"Original columns: {labeled_data.shape[1]}")
print(f"New columns added: {parsed_df.shape[1]}")
print(f"\nNew column names:")
print(parsed_df.columns.tolist())

labeled_data_expanded.head()

Original columns: 6
New columns added: 47

New column names:
['Patient Name', 'Age', 'Sex', 'Medical Record Number', 'Patient', 'Date of Admission', 'Date of Discharge', 'Patient name', 'Admission date', 'Discharge date', 'DOB', 'Admission Date', 'Discharge Date', 'Diagnosis', 'Gender', 'Hospital', 'Name', 'Hospital Number', 'Admitting Diagnosis', 'Admitted Service', 'Date of admission', 'Reason for admission', 'Final diagnosis', 'Treatment', 'Discharge instructions', 'Condition at discharge', 'Follow-up', 'Signed by', 'Principal Diagnosis', 'The patient, a 75-year-old Caucasian man, was admitted for bicytopenia in May 2013. His initial blood cell count showed hemoglobin 8 g/dl, platelets 87 × 109/l, and leukocytes 6.1 × 109/L. He had a medical history of T2D treated with biphasic insulin aspart and triple-bypass surgery for coronary disease in 2010. No family history of hematological malignancies was noted. He was hospitalized in an intensive care unit for grade IV anemia and then tra

,patient_identifier,text,has_cancer,has_diabetes,test_set,combined_label,Patient Name,Age,Sex,Medical Record Number,...,IV,Date of Birth,Discharge Diagnosis,Disposition,Attending Physician,"The patient, a 55-year-old female, was admitted to Mudanjiang Forestry Central Hospital with symptoms including dizziness, gait disturbance, and headache. Upon admission, the patient displayed a body temperature of 39.5 °C, blood pressure of 125/70 mm Hg, pulse rate of 60 beats/min, and respiration of 18 breaths/min. The neurological examination revealed moderate nuchal rigidity. Routine blood tests indicated an inflammatory response, with elevated neutrophil—granulocyte proportion and CRPs. An engorged adult tick had been removed from the patient two weeks before admission. Eleven days after tick removal, she had visited a local clinic with persistent high fever up to 42.0 °C and received supportive treatment with compound paracetamol tablets for two days with no clinical improvement. The routine test for the antibody of Borellia spp. and TBEV in cerebrospinal fluid (CSF) was negative for TBEV but positive for IgG antibody against Borrelia burgdorferi sensu lato (sl) at titer of 1",Hospital course,Autopsy findings,Diagnostic tests revealed a CTNNB1 hotspot class 5 pathogenic variant in exon 3,Patient Identifier
0,2200,DISCHARGE SUMMARY:\n\nPatient Name: [Redacted]...,1.0,0.0,0,Cancer Only,[Redacted],38,Female,[Redacted],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,645,Discharge Summary:\n\nPatient: [Name]\n\nMedic...,0.0,0.0,0,Neither,NaN,NaN,NaN,[Number],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2563,Discharge Summary:\nPatient name: [REDACTED]\n...,0.0,0.0,0,Neither,NaN,70 years,Female,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2275,Discharge Summary:\n\nPatient: 59-year-old Ita...,1.0,0.0,0,Cancer Only,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1828,Hospital Course:\n\nThe 80-year-old male prese...,0.0,1.0,0,Diabetes Only,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
# Check which headers appear most frequently and contain actual data
header_stats = parsed_df.notna().sum().sort_values(ascending=False)
print("Top 20 most populated headers:")
print(header_stats.head(20))

# Show a few sample values for top headers to see what's useful
print("\n\nSample values from top headers:")
for col in header_stats.head(10).index:
    sample_val = parsed_df[col].dropna().iloc[0] if parsed_df[col].notna().any() else "No data"
    print(f"\n{col}: {sample_val[:200]}...")  # First 200 chars

Top 20 most populated headers:
Patient Name             17
Age                      11
Admission Date           11
Discharge Date            9
Gender                    9
Medical Record Number     8
Date of Discharge         6
Date of Admission         6
Admitting Diagnosis       5
Patient                   4
Sex                       4
Name                      3
Follow-up                 3
Diagnosis                 3
Reason for admission      2
Discharge Diagnosis       2
Disposition               2
Date of Birth             2
DOB                       1
Patient name              1
dtype: int64


Sample values from top headers:

Patient Name: [Redacted]...

Age: 38...

Admission Date: [Redacted]...

Discharge Date: [Redacted]...

Gender: Female...

Medical Record Number: [Redacted]...

Date of Discharge: [Date]...

Date of Admission: [Date]...

Admitting Diagnosis: N/A...

Patient: [Name]...


In [98]:
# Keep only high-value columns (>1% coverage or >20 records)
high_value_headers = [
    # Demographics (good coverage, known risk factors)
    'Age', 'Gender', 'Sex',  # Keep both Gender and Sex, will merge later
    
    # Diagnosis fields (best signal for regex/LLM)
    'Diagnosis', 'Admitting Diagnosis', 'Discharge Diagnosis', 
    'Admission Diagnosis', 'Final Diagnosis',
    
    # Clinical narrative (rich text)
    'Hospital Course', 'History of Present Illness', 'Chief Complaint',
    'Medical History',
    
    # Medications (strong signal: metformin, chemotherapy, etc.)
    'Discharge Medications',
    
    # Treatment/Procedures
    'Treatment',
    
    # Follow-up
    'Follow-up'
]

# Start with original columns + only keep high-value parsed columns
original_cols = labeled_data.columns.tolist()
parsed_cols_to_keep = [col for col in high_value_headers if col in labeled_data_expanded.columns]

labeled_data_filtered = labeled_data_expanded[original_cols + parsed_cols_to_keep].copy()

# Merge Gender and Sex into single 'Gender' column
if 'Sex' in labeled_data_filtered.columns and 'Gender' in labeled_data_filtered.columns:
    labeled_data_filtered['Gender'] = labeled_data_filtered['Gender'].fillna(labeled_data_filtered['Sex'])
    labeled_data_filtered = labeled_data_filtered.drop('Sex', axis=1)
elif 'Sex' in labeled_data_filtered.columns:
    labeled_data_filtered = labeled_data_filtered.rename(columns={'Sex': 'Gender'})

# Show coverage
print("Final useful columns:")
useful_cols = [col for col in parsed_cols_to_keep if col in labeled_data_filtered.columns and col != 'Sex']
for col in useful_cols:
    count = labeled_data_filtered[col].notna().sum()
    pct = (count / len(labeled_data_filtered)) * 100
    print(f"  {col}: {count} records ({pct:.1f}%)")

print(f"\nTotal parsed columns: {len(useful_cols)}")
labeled_data_filtered.head()

Final useful columns:
  Age: 11 records (22.0%)
  Gender: 13 records (26.0%)
  Diagnosis: 3 records (6.0%)
  Admitting Diagnosis: 5 records (10.0%)
  Discharge Diagnosis: 2 records (4.0%)
  Chief Complaint: 1 records (2.0%)
  Treatment: 1 records (2.0%)
  Follow-up: 3 records (6.0%)

Total parsed columns: 8


,patient_identifier,text,has_cancer,has_diabetes,test_set,combined_label,Age,Gender,Diagnosis,Admitting Diagnosis,Discharge Diagnosis,Chief Complaint,Treatment,Follow-up
0,2200,DISCHARGE SUMMARY:\n\nPatient Name: [Redacted]...,1.0,0.0,0,Cancer Only,38,Female,NaN,NaN,NaN,NaN,NaN,NaN
1,645,Discharge Summary:\n\nPatient: [Name]\n\nMedic...,0.0,0.0,0,Neither,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2563,Discharge Summary:\nPatient name: [REDACTED]\n...,0.0,0.0,0,Neither,70 years,Female,NaN,NaN,NaN,NaN,NaN,NaN
3,2275,Discharge Summary:\n\nPatient: 59-year-old Ita...,1.0,0.0,0,Cancer Only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1828,Hospital Course:\n\nThe 80-year-old male prese...,0.0,1.0,0,Diabetes Only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
# Final EDA checks before modeling

print("="*60)
print("FINAL EDA SUMMARY")
print("="*60)

# 1. Label distribution by class
print("\n1. LABEL DISTRIBUTION:")
print(labeled_data_filtered['combined_label'].value_counts())
print(f"\nClass imbalance ratios:")
label_counts = labeled_data_filtered['combined_label'].value_counts()
for label in label_counts.index:
    ratio = label_counts[label] / len(labeled_data_filtered)
    print(f"  {label}: {ratio:.1%}")

# 2. Text length analysis by class
print("\n2. TEXT LENGTH BY CLASS:")
labeled_data_filtered['text_length'] = labeled_data_filtered['text'].str.len()
text_stats = labeled_data_filtered.groupby('combined_label')['text_length'].agg(['mean', 'median', 'min', 'max'])
print(text_stats)

# 3. Check for any obvious keyword patterns
print("\n3. KEYWORD PREVALENCE CHECK:")
keywords = {
    'diabetes': r'\bdiabetes|\bdiabetic|\bglucose|\binsulin|\bmetformin|\bHbA1c',
    'cancer': r'\bcancer|\bcarcinoma|\bmalignant|\bchemotherapy|\boncology|\btumor|\bneoplasm',
}

for keyword, pattern in keywords.items():
    labeled_data_filtered[f'has_{keyword}_keyword'] = labeled_data_filtered['text'].str.contains(pattern, case=False, regex=True, na=False)
    
print("\nKeyword matches by class:")
for keyword in keywords.keys():
    keyword_col = f'has_{keyword}_keyword'
    crosstab = pd.crosstab(labeled_data_filtered['combined_label'], labeled_data_filtered[keyword_col], normalize='index') * 100
    print(f"\n{keyword.upper()} keyword in text:")
    print(crosstab.round(1))

#Sample texts by class
print("\n5. SAMPLE TEXTS BY CLASS (first 300 chars):")
for label in labeled_data_filtered['combined_label'].unique():
    print(f"\n{label.upper()}:")
    sample = labeled_data_filtered[labeled_data_filtered['combined_label'] == label]['text'].iloc[0]
    print(f"  {sample[:300]}...")


FINAL EDA SUMMARY

1. LABEL DISTRIBUTION:
combined_label
Neither          27
Cancer Only      18
Diabetes Only     3
Both              2
Name: count, dtype: int64

Class imbalance ratios:
  Neither: 54.0%
  Cancer Only: 36.0%
  Diabetes Only: 6.0%
  Both: 4.0%

2. TEXT LENGTH BY CLASS:
                       mean  median   min   max
combined_label                                 
Both            1923.500000  1923.5  1674  2173
Cancer Only     1924.444444  1852.0  1177  2982
Diabetes Only   1613.666667  1644.0  1422  1775
Neither         1802.000000  1770.0  1076  2861

3. KEYWORD PREVALENCE CHECK:

Keyword matches by class:

DIABETES keyword in text:
has_diabetes_keyword  False  True 
combined_label                    
Both                    0.0  100.0
Cancer Only           100.0    0.0
Diabetes Only           0.0  100.0
Neither                85.2   14.8

CANCER keyword in text:
has_cancer_keyword  False  True 
combined_label                  
Both                 50.0   50.0
Cancer 

### Data Preparation

In [ ]:
# OpenAI embeddings - simpler, no local dependencies
import openai
import numpy as np
from tqdm.auto import tqdm

openai.api_key = ''  # Add your OpenAI key

def get_openai_embeddings(texts, model="text-embedding-3-small"):
    embeddings = []
    for i in tqdm(range(0, len(texts), 100)):  # Batch of 100
        batch = texts[i:i+100]
        response = openai.embeddings.create(
            input=batch,
            model=model
        )
        batch_embeddings = [item.embedding for item in response.data]
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)

# Generate embeddings
embeddings = get_openai_embeddings(df['text'].tolist())
df['embeddings'] = list(embeddings)
np.save('openai_embeddings.npy', embeddings)

100%|██████████| 20/20 [00:23<00:00,  1.19s/it]


In [101]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np
import pandas as pd

# ============================================
# VISUALIZE EMBEDDINGS OF LABELED DATA
# ============================================

print("="*80)
print("EMBEDDING VISUALIZATION - LABELED DATA ONLY")
print("="*80)

# Get labeled data only
labeled_data = df[(df['test_set'] == 0) & (df['has_cancer'].notna()) & (df['has_diabetes'].notna())].copy()
labeled_data['combined_label'] = labeled_data.apply(lambda row: get_combined_label(row), axis=1)

print(f"\nLabeled examples: {len(labeled_data)}")
print("\nClass distribution:")
print(labeled_data['combined_label'].value_counts())

# Extract embeddings (1536 dimensions)
X_labeled = np.vstack(labeled_data['embeddings'].values)
y_labeled = labeled_data['combined_label'].values

print(f"\nEmbedding shape: {X_labeled.shape}")

# ============================================
# METHOD 1: PCA (Linear dimensionality reduction)
# ============================================
print("\n" + "="*80)
print("PCA: Reducing to 2D (captures linear relationships)")
print("="*80)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_labeled)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.1%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.1%}")

# Create DataFrame for plotting
pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Label': y_labeled,
    'Patient_ID': labeled_data['patient_identifier'].values,
    'Text_Preview': [text[:100] + '...' for text in labeled_data['text'].values]
})

# Plot PCA
fig_pca = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Label',
    hover_data=['Patient_ID', 'Text_Preview'],
    title='PCA Projection of Patient Discharge Summaries (Labeled Data)',
    labels={'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)'},
    color_discrete_map={
        'Neither': '#636EFA',      # Blue
        'Cancer Only': '#EF553B',  # Red
        'Diabetes Only': '#00CC96', # Green
        'Both': '#AB63FA'          # Purple
    }
)

fig_pca.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
fig_pca.update_layout(
    width=1000,
    height=700,
    font=dict(size=12),
    legend=dict(
        title="Diagnosis",
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

fig_pca.show()

# ============================================
# METHOD 2: t-SNE (Non-linear dimensionality reduction)
# ============================================
print("\n" + "="*80)
print("t-SNE: Reducing to 2D (captures non-linear relationships)")
print("="*80)
print("Note: t-SNE can take a few minutes for larger datasets...")

tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(labeled_data)-1))
X_tsne = tsne.fit_transform(X_labeled)

# Create DataFrame for plotting
tsne_df = pd.DataFrame({
    'Dimension 1': X_tsne[:, 0],
    'Dimension 2': X_tsne[:, 1],
    'Label': y_labeled,
    'Patient_ID': labeled_data['patient_identifier'].values,
    'Text_Preview': [text[:100] + '...' for text in labeled_data['text'].values]
})

# Plot t-SNE
fig_tsne = px.scatter(
    tsne_df,
    x='Dimension 1',
    y='Dimension 2',
    color='Label',
    hover_data=['Patient_ID', 'Text_Preview'],
    title='t-SNE Projection of Patient Discharge Summaries (Labeled Data)',
    color_discrete_map={
        'Neither': '#636EFA',      # Blue
        'Cancer Only': '#EF553B',  # Red
        'Diabetes Only': '#00CC96', # Green
        'Both': '#AB63FA'          # Purple
    }
)

fig_tsne.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
fig_tsne.update_layout(
    width=1000,
    height=700,
    font=dict(size=12),
    legend=dict(
        title="Diagnosis",
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

fig_tsne.show()




# ============================================
# ANALYSIS: Class Separability
# ============================================
print("\n" + "="*80)
print("EMBEDDING SPACE ANALYSIS")
print("="*80)

# Calculate centroid distances between classes
from scipy.spatial.distance import cdist

class_centroids = {}
for label in ['Neither', 'Cancer Only', 'Diabetes Only', 'Both']:
    mask = y_labeled == label
    if mask.sum() > 0:
        class_centroids[label] = X_labeled[mask].mean(axis=0)

print("\nCentroid distances (cosine similarity):")
for i, label1 in enumerate(class_centroids.keys()):
    for label2 in list(class_centroids.keys())[i+1:]:
        distance = cdist([class_centroids[label1]], [class_centroids[label2]], metric='cosine')[0][0]
        similarity = 1 - distance
        print(f"  {label1:20s} <-> {label2:20s}: {similarity:.3f}")

# Intra-class variance
print("\nIntra-class variance (average distance to centroid):")
for label in class_centroids.keys():
    mask = y_labeled == label
    if mask.sum() > 0:
        distances = cdist(X_labeled[mask], [class_centroids[label]], metric='cosine')
        avg_distance = distances.mean()
        print(f"  {label:20s}: {avg_distance:.3f}")



EMBEDDING VISUALIZATION - LABELED DATA ONLY

Labeled examples: 50

Class distribution:
combined_label
Neither          27
Cancer Only      18
Diabetes Only     3
Both              2
Name: count, dtype: int64

Embedding shape: (50, 1536)

PCA: Reducing to 2D (captures linear relationships)
Explained variance: 14.8%
  PC1: 7.8%
  PC2: 7.0%



t-SNE: Reducing to 2D (captures non-linear relationships)
Note: t-SNE can take a few minutes for larger datasets...



EMBEDDING SPACE ANALYSIS

Centroid distances (cosine similarity):
  Neither              <-> Cancer Only         : 0.932
  Neither              <-> Diabetes Only       : 0.868
  Neither              <-> Both                : 0.768
  Cancer Only          <-> Diabetes Only       : 0.857
  Cancer Only          <-> Both                : 0.823
  Diabetes Only        <-> Both                : 0.743

Intra-class variance (average distance to centroid):
  Neither             : 0.273
  Cancer Only         : 0.232
  Diabetes Only       : 0.155
  Both                : 0.150


### Weak Supervised Learning (EDA)

In [102]:
import re
import numpy as np
import pandas as pd
from snorkel.labeling import labeling_function, PandasLFApplier
from snorkel.labeling.model import LabelModel

ABSTAIN = -1
NEG = 0
POS = 1

# ------------------------------
# Helpers
# ------------------------------
def has(pattern: str, text: str) -> bool:
    return re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE) is not None

# ------------------------------
# DIABETES labeling functions
# ------------------------------
@labeling_function()
def lf_dm_explicit(x):
    if has(r"\bdiabetes mellitus\b", x.text) or has(r"\btype\s*(1|2)\s*diabetes\b", x.text) or has(r"\bT2DM\b|\bT1DM\b", x.text):
        return POS
    return ABSTAIN

@labeling_function()
def lf_dm_insulin_dependent(x):
    if has(r"\binsulin[-\s]?dependent diabetes\b|\bIDDM\b", x.text):
        return POS
    return ABSTAIN

@labeling_function()
def lf_dm_complication(x):
    if has(r"\bdiabetic neuropathy\b|\bdiabetic retinopathy\b|\bdiabetic nephropathy\b", x.text):
        return POS
    return ABSTAIN

@labeling_function()
def lf_dm_negation(x):
    if has(r"\bno (history of )?diabetes\b|\bdenies diabetes\b", x.text):
        return NEG
    return ABSTAIN

@labeling_function()
def lf_diabetes_insipidus_guard(x):
    if has(r"\bdiabetes insipidus\b|\bDI\b", x.text):
        return NEG
    return ABSTAIN

DIABETES_LFS = [
    lf_dm_explicit,
    lf_dm_insulin_dependent,
    lf_dm_complication,
    lf_dm_negation,
    lf_diabetes_insipidus_guard,
]

# ------------------------------
# CANCER labeling functions
# ------------------------------
@labeling_function()
def lf_cancer_pathology_terms(x):
    if has(r"\bcarcinoma\b|\badenocarcinoma\b|\bsarcoma\b|\blymphoma\b|\bleukemia\b|\bmyeloma\b", x.text):
        return POS
    return ABSTAIN

@labeling_function()
def lf_cancer_metastatic(x):
    if has(r"\bmetastatic\b|\bmetastasis\b", x.text):
        return POS
    return ABSTAIN

@labeling_function()
def lf_cancer_history(x):
    if has(r"\bhistory of (.* )?cancer\b|\bhx of (.* )?cancer\b", x.text):
        return POS
    return ABSTAIN

@labeling_function()
def lf_cancer_negation(x):
    if has(r"\bno evidence of malignancy\b|\bmalignancy ruled out\b|\bbenign\b", x.text):
        return NEG
    return ABSTAIN

@labeling_function()
def lf_cancer_histiocytic(x):
    if has(r"\bLCH\b|\bLangerhans\b|\bhistiocytosis\b", x.text):
        return POS
    return ABSTAIN

CANCER_LFS = [
    lf_cancer_pathology_terms,
    lf_cancer_metastatic,
    lf_cancer_history,
    lf_cancer_negation,
    lf_cancer_histiocytic,
]

def fit_label_model(L: np.ndarray, n_epochs: int = 500, seed: int = 42) -> LabelModel:
    label_model = LabelModel(cardinality=2, verbose=False)
    label_model.fit(L_train=L, n_epochs=n_epochs, log_freq=100, seed=seed)
    return label_model

# ------------------------------
# Run Weak Supervision Analysis
# ------------------------------
print("WEAK SUPERVISION BASELINE ANALYSIS")
print("="*80)

# Combine all data for weak supervision
all_data = pd.concat([df_train, df_validation, df_unlabeled], ignore_index=True)
all_data['text'] = all_data['text'].fillna('').astype(str)

print(f"\nTotal rows: {len(all_data)}")
print(f"Training set: {len(df_train)}")
print(f"Validation set: {len(df_validation)}")
print(f"Unlabeled set: {len(df_unlabeled)}")

# Apply labeling functions
diabetes_applier = PandasLFApplier(lfs=DIABETES_LFS)
cancer_applier = PandasLFApplier(lfs=CANCER_LFS)

L_diabetes = diabetes_applier.apply(df=all_data)
L_cancer = cancer_applier.apply(df=all_data)

# Fit label models (unsupervised)
print("\nFitting label models...")
dm_model = fit_label_model(L_diabetes)
ca_model = fit_label_model(L_cancer)

# Generate probabilistic labels
dm_probs = dm_model.predict_proba(L_diabetes)[:, 1]
ca_probs = ca_model.predict_proba(L_cancer)[:, 1]

# Add to dataframe
all_data['ws_dm_prob'] = dm_probs
all_data['ws_cancer_prob'] = ca_probs

# Thresholds
DM_THRESH = 0.80
CA_THRESH = 0.80
all_data['ws_dm_label'] = (all_data['ws_dm_prob'] >= DM_THRESH).astype(int)
all_data['ws_cancer_label'] = (all_data['ws_cancer_prob'] >= CA_THRESH).astype(int)

# Evaluate on validation set
print("\n" + "="*80)
print("WEAK SUPERVISION PERFORMANCE ON VALIDATION SET")
print("="*80)

val_results = all_data.iloc[len(df_train):len(df_train)+len(df_validation)].copy()
val_results['true_dm'] = df_validation['has_diabetes'].values
val_results['true_cancer'] = df_validation['has_cancer'].values

dm_acc = (val_results['ws_dm_label'] == val_results['true_dm']).mean()
ca_acc = (val_results['ws_cancer_label'] == val_results['true_cancer']).mean()

print(f"\nDiabetes accuracy @ {DM_THRESH}: {dm_acc:.1%}")
print(f"Cancer accuracy @ {CA_THRESH}: {ca_acc:.1%}")

# Analyze uncertainty (0.5 means no LF fired)
uncertainty_mask = (all_data['ws_dm_prob'] == 0.5) & (all_data['ws_cancer_prob'] == 0.5)
print(f"\nRows with no signal (prob=0.5 for both): {uncertainty_mask.sum()} ({uncertainty_mask.sum()/len(all_data)*100:.1f}%)")

high_conf_dm = (all_data['ws_dm_prob'] > 0.8).sum()
high_conf_cancer = (all_data['ws_cancer_prob'] > 0.8).sum()
print(f"High confidence diabetes (>0.8): {high_conf_dm} ({high_conf_dm/len(all_data)*100:.1f}%)")
print(f"High confidence cancer (>0.8): {high_conf_cancer} ({high_conf_cancer/len(all_data)*100:.1f}%)")

print("\n" + "="*80)
print("EXAMPLE CASES")
print("="*80)

print("\nHigh confidence positive examples:")
high_conf = val_results[(val_results['ws_dm_prob'] > 0.9) | (val_results['ws_cancer_prob'] > 0.9)].head(3)
for idx, row in high_conf.iterrows():
    print(f"\nPatient {row['patient_identifier']}")
    print(f"  WS: DM={row['ws_dm_prob']:.3f}, Cancer={row['ws_cancer_prob']:.3f}")
    print(f"  True: DM={row['true_dm']}, Cancer={row['true_cancer']}")
    print(f"  Text preview: {row['text'][:200]}...")

print("\nNo signal examples (prob=0.5):")
no_signal = val_results[uncertainty_mask.iloc[len(df_train):len(df_train)+len(df_validation)]].head(3)
for idx, row in no_signal.iterrows():
    print(f"\nPatient {row['patient_identifier']}")
    print(f"  WS: DM={row['ws_dm_prob']:.3f}, Cancer={row['ws_cancer_prob']:.3f}")
    print(f"  True: DM={row['true_dm']}, Cancer={row['true_cancer']}")
    print(f"  Text preview: {row['text'][:200]}...")

WEAK SUPERVISION BASELINE ANALYSIS

Total rows: 1800
Training set: 40
Validation set: 10
Unlabeled set: 1750


100%|██████████| 1800/1800 [00:00<00:00, 5758.13it/s]



Fitting label models...


100%|██████████| 500/500 [00:00<00:00, 2836.31epoch/s]


WEAK SUPERVISION PERFORMANCE ON VALIDATION SET

Diabetes accuracy @ 0.8: 90.0%
Cancer accuracy @ 0.8: 90.0%

Rows with no signal (prob=0.5 for both): 1116 (62.0%)
High confidence diabetes (>0.8): 12 (0.7%)
High confidence cancer (>0.8): 547 (30.4%)

EXAMPLE CASES

High confidence positive examples:

Patient 1047
  WS: DM=0.500, Cancer=0.999
  True: DM=0.0, Cancer=1.0
  Text preview: Discharge Summary

Patient Name: [REDACTED]
Gender: Male
Age: 62 years
Medical Record Number: [REDACTED]

Hospital Course:
The patient was admitted to the hospital for the treatment of bullous hemorrh...

Patient 2117
  WS: DM=0.500, Cancer=1.000
  True: DM=0.0, Cancer=1.0
  Text preview: Hospital Course:

The patient is a 10-year-old girl who presented with a firm, painless swelling in the distal right forearm. X-ray examination revealed a large lobulated, compartmentalized, osteolyti...

Patient 1809
  WS: DM=0.500, Cancer=1.000
  True: DM=0.0, Cancer=1.0
  Text preview: Discharge Summary:

Patient Name:

### Multi Class Classification

In [103]:
from sklearn.model_selection import train_test_split

# Filter: test_set = 0 AND not null (labeled training data only)
labeled_data = df[(df['test_set'] == 0) & (df['has_cancer'].notna()) & (df['has_diabetes'].notna())].copy()

# Create combined label
def get_combined_label(row):
    has_cancer = row['has_cancer'] == 1
    has_diabetes = row['has_diabetes'] == 1
    
    if has_cancer and has_diabetes:
        return 'Both'
    elif has_cancer:
        return 'Cancer Only'
    elif has_diabetes:
        return 'Diabetes Only'
    else:
        return 'Neither'

labeled_data['combined_label'] = labeled_data.apply(get_combined_label, axis=1)

print(f"Total labeled records: {len(labeled_data)}")
print("\nClass distribution:")
print(labeled_data['combined_label'].value_counts())

# Stratified train/test split (80/20)
train_set, test_set = train_test_split(
    labeled_data,
    test_size=0.2,
    stratify=labeled_data['combined_label'],
    random_state=42
)

print(f"\nTrain: {len(train_set)} | Test: {len(test_set)}")
print("\nTrain class distribution:")
print(train_set['combined_label'].value_counts())
print("\nTest class distribution:")
print(test_set['combined_label'].value_counts())

# Extract embeddings as numpy arrays
X_train = np.vstack(train_set['embeddings'].values)
X_test = np.vstack(test_set['embeddings'].values)

# Extract labels
y_train = train_set['combined_label'].values
y_test = test_set['combined_label'].values

# Binary labels
y_train_diabetes = (train_set['has_diabetes'] == 1).astype(int).values
y_train_cancer = (train_set['has_cancer'] == 1).astype(int).values
y_test_diabetes = (test_set['has_diabetes'] == 1).astype(int).values
y_test_cancer = (test_set['has_cancer'] == 1).astype(int).values

print(f"\nEmbedding shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

print(f"\nBinary label counts:")
print(f"Train - Diabetes: {y_train_diabetes.sum()}/{len(y_train_diabetes)}")
print(f"Train - Cancer: {y_train_cancer.sum()}/{len(y_train_cancer)}")
print(f"Test - Diabetes: {y_test_diabetes.sum()}/{len(y_test_diabetes)}")
print(f"Test - Cancer: {y_test_cancer.sum()}/{len(y_test_cancer)}")

Total labeled records: 50

Class distribution:
combined_label
Neither          27
Cancer Only      18
Diabetes Only     3
Both              2
Name: count, dtype: int64

Train: 40 | Test: 10

Train class distribution:
combined_label
Neither          22
Cancer Only      14
Both              2
Diabetes Only     2
Name: count, dtype: int64

Test class distribution:
combined_label
Neither          5
Cancer Only      4
Diabetes Only    1
Name: count, dtype: int64

Embedding shapes:
X_train: (40, 1536)
X_test: (10, 1536)

Binary label counts:
Train - Diabetes: 4/40
Train - Cancer: 16/40
Test - Diabetes: 1/10
Test - Cancer: 4/10


In [104]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import plotly.figure_factory as ff


print("="*60)
print("MULTI-CLASS CLASSIFICATION - DIABETES vs CANCER vs BOTH vs NEITHER")
print("="*60)

# 1. WITH class weighting (regularization for imbalance)
print("\n1. WITH CLASS BALANCING (class_weight='balanced'):")
print("-"*60)

clf_multiclass_balanced = LogisticRegression(
    class_weight='balanced',  # Regularization penalty for imbalance
    C=1.0,
    max_iter=1000,
    random_state=42
)
clf_multiclass_balanced.fit(X_train, y_train)

# Predictions
y_pred_balanced = clf_multiclass_balanced.predict(X_test)
y_pred_proba_balanced = clf_multiclass_balanced.predict_proba(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced, zero_division=0))

print("\nConfusion Matrix:")
cm_balanced = confusion_matrix(y_test, y_pred_balanced, labels=['Neither', 'Cancer Only', 'Diabetes Only', 'Both'])
cm_df_balanced = pd.DataFrame(cm_balanced, 
                               index=['True: Neither', 'True: Cancer Only', 'True: Diabetes Only', 'True: Both'],
                               columns=['Pred: Neither', 'Pred: Cancer Only', 'Pred: Diabetes Only', 'Pred: Both'])

fig1 = ff.create_annotated_heatmap(
    z=cm_balanced,
    x=['Pred: Neither', 'Pred: Cancer Only', 'Pred: Diabetes Only', 'Pred: Both'],
    y=['True: Neither', 'True: Cancer Only', 'True: Diabetes Only', 'True: Both'],
    colorscale='Blues',
    showscale=True
)
fig1.update_layout(
    title='Confusion Matrix - WITH Class Balancing',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    height=500
)
fig1.show()



print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred_balanced):.3f}")

# 2. WITHOUT class weighting
print("\n\n2. WITHOUT CLASS BALANCING (no class_weight):")
print("-"*60)

clf_multiclass_no_balance = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)
clf_multiclass_no_balance.fit(X_train, y_train)

# Predictions
y_pred_no_balance = clf_multiclass_no_balance.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_no_balance, zero_division=0))

print("\nConfusion Matrix:")
cm_no_balance = confusion_matrix(y_test, y_pred_no_balance, labels=['Neither', 'Cancer Only', 'Diabetes Only', 'Both'])
cm_df_no_balance = pd.DataFrame(cm_no_balance,
                                 index=['True: Neither', 'True: Cancer Only', 'True: Diabetes Only', 'True: Both'],
                                 columns=['Pred: Neither', 'Pred: Cancer Only', 'Pred: Diabetes Only', 'Pred: Both'])
fig2 = ff.create_annotated_heatmap(
    z=cm_no_balance,
    x=['Pred: Neither', 'Pred: Cancer Only', 'Pred: Diabetes Only', 'Pred: Both'],
    y=['True: Neither', 'True: Cancer Only', 'True: Diabetes Only', 'True: Both'],
    colorscale='Reds',
    showscale=True
)
fig2.update_layout(
    title='Confusion Matrix - WITHOUT Class Balancing',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    height=500
)
fig2.show()

print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred_no_balance):.3f}")

print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"With Balancing: {accuracy_score(y_test, y_pred_balanced):.3f}")
print(f"Without Balancing: {accuracy_score(y_test, y_pred_no_balance):.3f}")


MULTI-CLASS CLASSIFICATION - DIABETES vs CANCER vs BOTH vs NEITHER

1. WITH CLASS BALANCING (class_weight='balanced'):
------------------------------------------------------------

Classification Report:
               precision    recall  f1-score   support

  Cancer Only       0.67      0.50      0.57         4
Diabetes Only       0.00      0.00      0.00         1
      Neither       0.71      1.00      0.83         5

     accuracy                           0.70        10
    macro avg       0.46      0.50      0.47        10
 weighted avg       0.62      0.70      0.65        10


Confusion Matrix:



Overall Accuracy: 0.700


2. WITHOUT CLASS BALANCING (no class_weight):
------------------------------------------------------------

Classification Report:
               precision    recall  f1-score   support

  Cancer Only       1.00      0.25      0.40         4
Diabetes Only       0.00      0.00      0.00         1
      Neither       0.56      1.00      0.71         5

     accuracy                           0.60        10
    macro avg       0.52      0.42      0.37        10
 weighted avg       0.68      0.60      0.52        10


Confusion Matrix:



Overall Accuracy: 0.600

COMPARISON
With Balancing: 0.700
Without Balancing: 0.600


### Semi- Supervised Approach

In [105]:
print("="*60)
print("TRAINING BASELINE BINARY CLASSIFIERS")
print("="*60)

from sklearn.linear_model import LogisticRegression

# Extract training embeddings
X_train = np.vstack(train_set['embeddings'].values)

# Binary labels for diabetes and cancer
y_train_diabetes = train_set['has_diabetes'].values
y_train_cancer = train_set['has_cancer'].values

# Train binary classifiers with class balancing
clf_diabetes_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
clf_diabetes_balanced.fit(X_train, y_train_diabetes)

clf_cancer_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
clf_cancer_balanced.fit(X_train, y_train_cancer)

# Check max confidence on training data (to understand model strength)
train_prob_diabetes = clf_diabetes_balanced.predict_proba(X_train)[:, 1]
train_prob_cancer = clf_cancer_balanced.predict_proba(X_train)[:, 1]

print(f"\nDiabetes classifier:")
print(f"  Max confidence: {train_prob_diabetes.max():.1%}")
print(f"  Mean confidence (positive class): {train_prob_diabetes[y_train_diabetes==1].mean():.1%}")
print(f"  Mean confidence (negative class): {train_prob_diabetes[y_train_diabetes==0].mean():.1%}")

print(f"\nCancer classifier:")
print(f"  Max confidence: {train_prob_cancer.max():.1%}")
print(f"  Mean confidence (positive class): {train_prob_cancer[y_train_cancer==1].mean():.1%}")
print(f"  Mean confidence (negative class): {train_prob_cancer[y_train_cancer==0].mean():.1%}")

print("\n✓ Baseline classifiers trained")
print("="*60)

TRAINING BASELINE BINARY CLASSIFIERS

Diabetes classifier:
  Max confidence: 64.1%
  Mean confidence (positive class): 62.5%
  Mean confidence (negative class): 37.5%

Cancer classifier:
  Max confidence: 65.2%
  Mean confidence (positive class): 58.3%
  Mean confidence (negative class): 41.8%

✓ Baseline classifiers trained


In [106]:
print("="*60)
print("PHASE 1: ML-BASED SYNTHETIC LABELING (REVISED)")
print("="*60)
print("NOTE: Using 60% threshold instead of 90% due to weak baseline models")
print("(Baseline max confidence: 64.8% diabetes, 65.4% cancer)")

# Get unlabeled data (test_set=0 and labels are null)
unlabeled_data = df[(df['test_set'] == 0) & (df['has_cancer'].isna()) & (df['has_diabetes'].isna())].copy()

print(f"\nStarting labeled training set: {len(train_set)}")
print(f"Available unlabeled data: {len(unlabeled_data)}")

# Extract unlabeled embeddings
X_unlabeled = np.vstack(unlabeled_data['embeddings'].values)

print("\nUsing baseline binary classifiers to predict on unlabeled data...")

# Get probability predictions
prob_diabetes = clf_diabetes_balanced.predict_proba(X_unlabeled)[:, 1]
prob_cancer = clf_cancer_balanced.predict_proba(X_unlabeled)[:, 1]

# Apply 60% confidence threshold (lowered from 90% due to weak baseline)
high_conf_diabetes = prob_diabetes >= 0.6
high_conf_cancer = prob_cancer >= 0.6
high_conf_neither = (prob_diabetes < 0.2) & (prob_cancer < 0.2)  # Very low for neither

# Combine: keep records with high confidence in ANY direction
high_conf_mask = high_conf_diabetes | high_conf_cancer | high_conf_neither

print(f"\nConfident predictions (≥60% positive or ≤20% negative):")
print(f"  Diabetes: {high_conf_diabetes.sum()}")
print(f"  Cancer: {high_conf_cancer.sum()}")
print(f"  Neither: {high_conf_neither.sum()}")
print(f"  Total: {high_conf_mask.sum()}")

if high_conf_mask.sum() > 0:
    # Create ML synthetic labels
    ml_synthetic = unlabeled_data[high_conf_mask].copy()
    ml_synthetic['has_diabetes'] = (prob_diabetes[high_conf_mask] >= 0.6).astype(int)
    ml_synthetic['has_cancer'] = (prob_cancer[high_conf_mask] >= 0.6).astype(int)
    ml_synthetic['combined_label'] = ml_synthetic.apply(get_combined_label, axis=1)
    ml_synthetic['source'] = 'ml_synthetic'
    
    print(f"\nML synthetic labels created: {len(ml_synthetic)}")
    print("\nML synthetic label distribution:")
    print(ml_synthetic['combined_label'].value_counts())
else:
    ml_synthetic = pd.DataFrame()
    print("\nNo confident predictions found - skipping ML synthetic labeling")

print(f"\n{'='*60}")
print("Phase 1 complete - proceeding to Phase 2 (Regex)")
print(f"{'='*60}")

PHASE 1: ML-BASED SYNTHETIC LABELING (REVISED)
NOTE: Using 60% threshold instead of 90% due to weak baseline models
(Baseline max confidence: 64.8% diabetes, 65.4% cancer)

Starting labeled training set: 40
Available unlabeled data: 1750

Using baseline binary classifiers to predict on unlabeled data...

Confident predictions (≥60% positive or ≤20% negative):
  Diabetes: 0
  Cancer: 142
  Neither: 0
  Total: 142

ML synthetic labels created: 142

ML synthetic label distribution:
combined_label
Cancer Only    142
Name: count, dtype: int64

Phase 1 complete - proceeding to Phase 2 (Regex)


In [107]:
# Examine actual diabetes cases to build targeted patterns
print("="*60)
print("ANALYZING TRUE DIABETES CASES FOR PATTERNS")
print("="*60)

# Get all labeled diabetes cases (from full df)
diabetes_cases = df[(df['has_diabetes'] == 1) & (df['test_set'] == 0)].copy()

print(f"Total diabetes cases: {len(diabetes_cases)}")
print("\nSample diabetes texts (first 500 chars each):\n")

for idx, row in diabetes_cases.iterrows():
    print(f"Case {idx}:")
    print(row['text'][:500])
    print("\n" + "-"*60 + "\n")

ANALYZING TRUE DIABETES CASES FOR PATTERNS
Total diabetes cases: 5

Sample diabetes texts (first 500 chars each):

Case 4:
Hospital Course:

The 80-year-old male presented to the emergency department with hematemesis and bleeding around the gastrostomy site. A review of his medication list showed he was taking aspirin 81 mg daily but no additional antiplatelet drugs, anticoagulants, or nonsteroidal anti-inflammatory drugs (NSAIDs). On physical exam, he was hypotensive, tachycardic, and febrile. The gastric ulcer with signs of recent bleeding was revealed as the cause of his gastrointestinal bleeding on EGD, which al

------------------------------------------------------------

Case 18:
Discharge Summary

Admission Date: May 2013 
Discharge Date: July 2013 
Principal Diagnosis: MDS with excess blasts: refractory anemia with excess blasts-1 (RAEB-1)

Hospital Course:

The patient, a 75-year-old Caucasian man, was admitted for bicytopenia in May 2013. His initial blood cell count showed 

In [108]:
print("="*60)
print("PHASE 2: REGEX-BASED SYNTHETIC LABELING")
print("="*60)

# Get remaining unlabeled data (exclude Phase 1 ML labels)
ml_synthetic_indices = ml_synthetic.index
remaining_after_ml = unlabeled_data[~unlabeled_data.index.isin(ml_synthetic_indices)].copy()

print(f"Remaining unlabeled after Phase 1: {len(remaining_after_ml)}")

# STRICT DIABETES PATTERNS (based on actual diabetes cases)
diabetes_patterns = (
    r'\bT2DM\b|\bT1DM\b|'
    r'\btype\s*[12]\s*diabetes(?:\s*mellitus)?\b|'
    r'\bdiabetes\s*mellitus\b|'
    r'\bhistory\s*of\s*diabetes\b|'
    r'\bdiabetic\b|'
    r'\binsulin\s*(?:aspart|therapy|dependent|glargine|lispro)\b|'
    r'\bmetformin\b|\bglipizide\b|\bglyburide\b'
)

# STRICT CANCER PATTERNS
cancer_patterns = (
    r'\bcancer\b|\bcarcinoma\b|\bsarcoma\b|\bmelanoma\b|'
    r'\blymphoma\b|\bleukemia\b|\bmalignancy\b|\bmalignant\b|'
    r'\bneoplasm\b|\bchemotherapy\b|\boncolog(?:y|ist)\b|'
    r'\bmetastatic\b|\btumor\b|\bMDS\s*with\s*excess\s*blasts\b'
)

print("\nApplying regex patterns to remaining data...")

# Apply regex
remaining_after_ml['has_diabetes_regex'] = remaining_after_ml['text'].str.contains(
    diabetes_patterns, case=False, regex=True, na=False
).astype(int)

remaining_after_ml['has_cancer_regex'] = remaining_after_ml['text'].str.contains(
    cancer_patterns, case=False, regex=True, na=False
).astype(int)

# Count matches
regex_diabetes = remaining_after_ml['has_diabetes_regex'].sum()
regex_cancer = remaining_after_ml['has_cancer_regex'].sum()
regex_both = ((remaining_after_ml['has_diabetes_regex'] == 1) & 
              (remaining_after_ml['has_cancer_regex'] == 1)).sum()

print(f"\nREGEX MATCHES:")
print(f"  Diabetes keywords: {regex_diabetes}")
print(f"  Cancer keywords: {regex_cancer}")
print(f"  Both keywords: {regex_both}")

# Create regex synthetic labels
regex_synthetic = remaining_after_ml[
    (remaining_after_ml['has_diabetes_regex'] == 1) | 
    (remaining_after_ml['has_cancer_regex'] == 1)
].copy()

regex_synthetic['has_diabetes'] = regex_synthetic['has_diabetes_regex']
regex_synthetic['has_cancer'] = regex_synthetic['has_cancer_regex']
regex_synthetic['combined_label'] = regex_synthetic.apply(get_combined_label, axis=1)
regex_synthetic['source'] = 'regex'

print(f"\nRegex synthetic labels: {len(regex_synthetic)}")
print("\nRegex label distribution:")
print(regex_synthetic['combined_label'].value_counts())

print(f"\n{'='*60}")
print("COMBINED: PHASE 1 (ML) + PHASE 2 (REGEX)")
print(f"{'='*60}")

# Save original training set for later use
train_set_original = train_set.copy()

# Combine labeled + Phase 1 ML + Phase 2 Regex
train_set_original['source'] = 'labeled'

synthetic_combined = pd.concat([
    train_set_original,
    ml_synthetic,
    regex_synthetic
], ignore_index=True)

print(f"Total size: {len(synthetic_combined)}")
print(f"  Original labeled: {len(train_set_original)}")
print(f"  Phase 1 (ML): {len(ml_synthetic)}")
print(f"  Phase 2 (Regex): {len(regex_synthetic)}")

print("\nCombined class distribution:")
print(synthetic_combined['combined_label'].value_counts())

print("\nBreakdown by source:")
print(pd.crosstab(synthetic_combined['combined_label'], synthetic_combined['source']))

# Prepare for Phase 3
X_synthetic_combined = np.vstack(synthetic_combined['embeddings'].values)
y_synthetic_combined = synthetic_combined['combined_label'].values

print(f"\nCombined embeddings shape: {X_synthetic_combined.shape}")
print(f"\nDiabetes examples: {(y_synthetic_combined == 'Diabetes Only').sum() + (y_synthetic_combined == 'Both').sum()}")
print(f"Cancer examples: {(y_synthetic_combined == 'Cancer Only').sum() + (y_synthetic_combined == 'Both').sum()}")
print(f"Neither examples: {(y_synthetic_combined == 'Neither').sum()}")

print(f"\n{'='*60}")
print("Phase 2 complete - proceeding to Phase 3 (Add 'Neither')")
print(f"{'='*60}")

PHASE 2: REGEX-BASED SYNTHETIC LABELING
Remaining unlabeled after Phase 1: 1608

Applying regex patterns to remaining data...

REGEX MATCHES:
  Diabetes keywords: 122
  Cancer keywords: 635
  Both keywords: 32

Regex synthetic labels: 725

Regex label distribution:
combined_label
Cancer Only      603
Diabetes Only     90
Both              32
Name: count, dtype: int64

COMBINED: PHASE 1 (ML) + PHASE 2 (REGEX)
Total size: 907
  Original labeled: 40
  Phase 1 (ML): 142
  Phase 2 (Regex): 725

Combined class distribution:
combined_label
Cancer Only      759
Diabetes Only     92
Both              34
Neither           22
Name: count, dtype: int64

Breakdown by source:
source          labeled  ml_synthetic  regex
combined_label                              
Both                  2             0     32
Cancer Only          14           142    603
Diabetes Only         2             0     90
Neither              22             0      0

Combined embeddings shape: (907, 1536)

Diabetes examples:

In [109]:
print("="*60)
print("PHASE 3: ADDING 'NEITHER' SYNTHETIC LABELS (REVISED)")
print("="*60)

print("ISSUE: Dataset heavily imbalanced toward positive cases")
print("  Cancer: 897 examples")
print("  Diabetes: 117 examples")
print("  Neither: 22 examples (only from labeled!)")
print("\nSOLUTION: Add synthetic 'Neither' examples for balance\n")

# Get remaining unlabeled data (exclude Phase 1 & 2)
used_indices = pd.concat([ml_synthetic, regex_synthetic]).index
remaining_after_phase2 = unlabeled_data[~unlabeled_data.index.isin(used_indices)].copy()

print(f"Remaining unlabeled after Phase 1 & 2: {len(remaining_after_phase2)}")

if len(remaining_after_phase2) > 0:
    X_remaining_neither = np.vstack(remaining_after_phase2['embeddings'].values)
    
    # Predict with baseline models
    prob_diabetes_neither = clf_diabetes_balanced.predict_proba(X_remaining_neither)[:, 1]
    prob_cancer_neither = clf_cancer_balanced.predict_proba(X_remaining_neither)[:, 1]
    
    print(f"\nProbability distribution in remaining data:")
    print(f"  Max diabetes prob: {prob_diabetes_neither.max():.3f}")
    print(f"  Max cancer prob: {prob_cancer_neither.max():.3f}")
    print(f"  Records with diabetes <30%: {(prob_diabetes_neither < 0.3).sum()}")
    print(f"  Records with cancer <30%: {(prob_cancer_neither < 0.3).sum()}")
    
    # RELAXED threshold: both diseases <30% (was 20%)
    neither_mask = (prob_diabetes_neither < 0.3) & (prob_cancer_neither < 0.3)
    
    print(f"\nHigh-confidence 'Neither' candidates (≤30% for both): {neither_mask.sum()}")
    
    if neither_mask.sum() > 0:
        # Take a sample to balance (target ~200)
        target_neither_count = min(neither_mask.sum(), 200)
        
        ml_synthetic_neither = remaining_after_phase2[neither_mask].sample(
            n=target_neither_count,
            random_state=42
        ).copy()
        
        ml_synthetic_neither['has_diabetes'] = 0
        ml_synthetic_neither['has_cancer'] = 0
        ml_synthetic_neither['combined_label'] = 'Neither'
        ml_synthetic_neither['source'] = 'ml_neither'
        
        print(f"Adding {len(ml_synthetic_neither)} synthetic 'Neither' examples")
        
        # Create FINAL BALANCED dataset
        final_expanded_balanced = pd.concat([
            train_set_original,
            ml_synthetic,
            regex_synthetic,
            ml_synthetic_neither
        ], ignore_index=True)
        
        print(f"\n{'='*60}")
        print("FINAL BALANCED DATASET (ALL 3 PHASES)")
        print(f"{'='*60}")
        print(f"Total size: {len(final_expanded_balanced)}")
        print(f"  Original labeled: {len(train_set_original)}")
        print(f"  Phase 1 (ML positive): {len(ml_synthetic)}")
        print(f"  Phase 2 (Regex): {len(regex_synthetic)}")
        print(f"  Phase 3 (ML neither): {len(ml_synthetic_neither)}")
        
        print("\nFinal balanced class distribution:")
        print(final_expanded_balanced['combined_label'].value_counts())
        
        print("\nBreakdown by source:")
        print(pd.crosstab(final_expanded_balanced['combined_label'], final_expanded_balanced['source']))
        
        # Prepare final training data
        X_final_balanced = np.vstack(final_expanded_balanced['embeddings'].values)
        y_final_balanced = final_expanded_balanced['combined_label'].values
        
        print(f"\nFinal balanced embeddings shape: {X_final_balanced.shape}")
        print(f"\nClass counts:")
        print(f"  Diabetes: {(y_final_balanced == 'Diabetes Only').sum() + (y_final_balanced == 'Both').sum()}")
        print(f"  Cancer: {(y_final_balanced == 'Cancer Only').sum() + (y_final_balanced == 'Both').sum()}")
        print(f"  Neither: {(y_final_balanced == 'Neither').sum()}")
        
        print(f"\n{'='*60}")
        print("SEMI-SUPERVISED LABELING COMPLETE")
        print(f"{'='*60}")
        print("✓ Phase 1: Added", len(ml_synthetic), "ML-based positive cases")
        print("✓ Phase 2: Added", len(regex_synthetic), "regex-based positive cases")
        print("✓ Phase 3: Added", len(ml_synthetic_neither), "'Neither' cases for balance")
        print(f"✓ Total expansion: 40 → {len(final_expanded_balanced)} examples")
        
    else:
        print("\nSTILL no 'Neither' candidates - baseline model cannot confidently predict 'Neither'")
        print("Proceeding with imbalanced dataset (will use class weighting in training)")
        final_expanded_balanced = synthetic_combined
        X_final_balanced = X_synthetic_combined
        y_final_balanced = y_synthetic_combined
        ml_synthetic_neither = pd.DataFrame()
else:
    print("No remaining unlabeled data")
    final_expanded_balanced = synthetic_combined
    X_final_balanced = X_synthetic_combined
    y_final_balanced = y_synthetic_combined
    ml_synthetic_neither = pd.DataFrame()

PHASE 3: ADDING 'NEITHER' SYNTHETIC LABELS (REVISED)
ISSUE: Dataset heavily imbalanced toward positive cases
  Cancer: 897 examples
  Diabetes: 117 examples
  Neither: 22 examples (only from labeled!)

SOLUTION: Add synthetic 'Neither' examples for balance

Remaining unlabeled after Phase 1 & 2: 883

Probability distribution in remaining data:
  Max diabetes prob: 0.548
  Max cancer prob: 0.597
  Records with diabetes <30%: 5
  Records with cancer <30%: 0

High-confidence 'Neither' candidates (≤30% for both): 0

STILL no 'Neither' candidates - baseline model cannot confidently predict 'Neither'
Proceeding with imbalanced dataset (will use class weighting in training)


In [110]:
print("\n" + "="*60)
print("FINAL COMPREHENSIVE EVALUATION")
print("="*60)
print("Dataset: 40 labeled + 979 synthetic (imbalanced)")
print("Strategy: Use class_weight='balanced' to handle imbalance")
print("="*60)

# Use the comprehensive test split from earlier
# (Same split that had 2 diabetes in test set)

# Get ALL 50 labeled examples
all_labeled = df[(df['test_set'] == 0) & (df['has_cancer'].notna()) & (df['has_diabetes'].notna())].copy()
all_labeled['combined_label'] = all_labeled.apply(get_combined_label, axis=1)

# Create test set with oversampling of rare classes
test_samples = []
train_samples = []

sampling_strategy = {
    'Neither': 0.25,
    'Cancer Only': 0.30,  
    'Diabetes Only': 0.67,
    'Both': 0.50
}

for label, test_fraction in sampling_strategy.items():
    label_data = all_labeled[all_labeled['combined_label'] == label]
    n_total = len(label_data)
    n_test = max(1, int(n_total * test_fraction))
    
    label_shuffled = label_data.sample(frac=1, random_state=42)
    test_samples.append(label_shuffled.iloc[:n_test])
    train_samples.append(label_shuffled.iloc[n_test:])

train_labeled = pd.concat(train_samples, ignore_index=True)
test_labeled = pd.concat(test_samples, ignore_index=True)

X_test_final = np.vstack(test_labeled['embeddings'].values)
y_test_final = test_labeled['combined_label'].values

print(f"\nTest set: {len(test_labeled)} examples")
print("Test distribution:", test_labeled['combined_label'].value_counts().to_dict())

# Get synthetic data
synthetic_only = final_expanded_balanced[final_expanded_balanced['source'] != 'labeled'].copy()
X_synthetic_only = np.vstack(synthetic_only['embeddings'].values)
y_synthetic_only = synthetic_only['combined_label'].values

# Combine train_labeled + synthetic
X_train_labeled_final = np.vstack(train_labeled['embeddings'].values)
y_train_labeled_final = train_labeled['combined_label'].values

X_train_combined_final = np.vstack([X_train_labeled_final, X_synthetic_only])
y_train_combined_final = np.concatenate([y_train_labeled_final, y_synthetic_only])

print(f"\nTraining set: {len(y_train_combined_final)} examples")
print(f"  Labeled: {len(train_labeled)}")
print(f"  Synthetic: {len(synthetic_only)}")
print("\nTraining distribution:", pd.Series(y_train_combined_final).value_counts().to_dict())

# MODEL 1: Baseline (labeled only)
print("\n" + "="*60)
print("MODEL 1: BASELINE (LABELED ONLY)")
print("="*60)

clf_baseline_final = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42)
clf_baseline_final.fit(X_train_labeled_final, y_train_labeled_final)
y_pred_baseline_final = clf_baseline_final.predict(X_test_final)

print("\nBaseline Performance:")
print(classification_report(y_test_final, y_pred_baseline_final, zero_division=0))

baseline_acc_final = accuracy_score(y_test_final, y_pred_baseline_final)

# MODEL 2: Semi-Supervised (labeled + synthetic)
print("\n" + "="*60)
print("MODEL 2: SEMI-SUPERVISED (LABELED + SYNTHETIC)")
print("="*60)

clf_semisup_final = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42)
clf_semisup_final.fit(X_train_combined_final, y_train_combined_final)
y_pred_semisup_final = clf_semisup_final.predict(X_test_final)

print("\nSemi-Supervised Performance:")
print(classification_report(y_test_final, y_pred_semisup_final, zero_division=0))

semisup_acc_final = accuracy_score(y_test_final, y_pred_semisup_final)

# Comparison
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)

comparison_data = []
for cls in ['Neither', 'Cancer Only', 'Diabetes Only', 'Both']:
    true_count = (y_test_final == cls).sum()
    if true_count > 0:
        baseline_correct = ((y_test_final == cls) & (y_pred_baseline_final == cls)).sum()
        baseline_recall = baseline_correct / true_count
        
        semisup_correct = ((y_test_final == cls) & (y_pred_semisup_final == cls)).sum()
        semisup_recall = semisup_correct / true_count
        
        comparison_data.append({
            'Class': cls,
            'Count': true_count,
            'Baseline': f"{baseline_recall:.1%}",
            'Semi-Supervised': f"{semisup_recall:.1%}",
            'Change': f"{(semisup_recall - baseline_recall)*100:+.0f}%"
        })

print("\n", pd.DataFrame(comparison_data).to_string(index=False))

print(f"\nOverall Accuracy:")
print(f"  Baseline: {baseline_acc_final:.1%}")
print(f"  Semi-Supervised: {semisup_acc_final:.1%}")
print(f"  Difference: {(semisup_acc_final - baseline_acc_final)*100:+.1f} pp")


FINAL COMPREHENSIVE EVALUATION
Dataset: 40 labeled + 979 synthetic (imbalanced)
Strategy: Use class_weight='balanced' to handle imbalance

Test set: 14 examples
Test distribution: {'Neither': 6, 'Cancer Only': 5, 'Diabetes Only': 2, 'Both': 1}

Training set: 903 examples
  Labeled: 36
  Synthetic: 867

Training distribution: {'Cancer Only': 758, 'Diabetes Only': 91, 'Both': 33, 'Neither': 21}

MODEL 1: BASELINE (LABELED ONLY)

Baseline Performance:
               precision    recall  f1-score   support

         Both       0.00      0.00      0.00         1
  Cancer Only       0.60      0.60      0.60         5
Diabetes Only       0.00      0.00      0.00         2
      Neither       0.56      0.83      0.67         6

     accuracy                           0.57        14
    macro avg       0.29      0.36      0.32        14
 weighted avg       0.45      0.57      0.50        14


MODEL 2: SEMI-SUPERVISED (LABELED + SYNTHETIC)

Semi-Supervised Performance:
               precision 

In [111]:
import re
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ============================================
# COMPLETE STANDALONE ENSEMBLE PIPELINE
# ============================================

print("="*80)
print("FINAL ENSEMBLE: COMPLETE STANDALONE PIPELINE")
print("="*80)

# ============================================
# STEP 1: CREATE STRATIFIED TRAIN/TEST SPLIT
# ============================================


# Get ALL labeled examples
all_labeled = df[(df['test_set'] == 0) & (df['has_cancer'].notna()) & (df['has_diabetes'].notna())].copy()
all_labeled['combined_label'] = all_labeled.apply(get_combined_label, axis=1)


# Create stratified test set with oversampling of rare classes
test_samples = []
train_samples = []

sampling_strategy = {
    'Neither': 0.25,
    'Cancer Only': 0.30,  
    'Diabetes Only': 0.67,
    'Both': 0.50
}

for label, test_fraction in sampling_strategy.items():
    label_data = all_labeled[all_labeled['combined_label'] == label]
    n_total = len(label_data)
    n_test = max(1, int(n_total * test_fraction))
    
    label_shuffled = label_data.sample(frac=1, random_state=42)
    test_samples.append(label_shuffled.iloc[:n_test])
    train_samples.append(label_shuffled.iloc[n_test:])
    

train_labeled = pd.concat(train_samples, ignore_index=True)
test_labeled = pd.concat(test_samples, ignore_index=True)

# ============================================
# STEP 2: TRAIN BINARY CLASSIFIERS ON TRAIN_LABELED
# ============================================

# Extract training embeddings from train_labeled
X_train_for_binary = np.vstack(train_labeled['embeddings'].values)
y_train_diabetes_binary = train_labeled['has_diabetes'].values
y_train_cancer_binary = train_labeled['has_cancer'].values

# Train binary classifiers with class balancing
clf_diabetes_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
clf_diabetes_balanced.fit(X_train_for_binary, y_train_diabetes_binary)

clf_cancer_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
clf_cancer_balanced.fit(X_train_for_binary, y_train_cancer_binary)

print("✓ Binary classifiers trained on train_labeled")

# ============================================
# STEP 3: PHASE 1 - ML-BASED SYNTHETIC LABELING
# ============================================

# Get unlabeled data (test_set=0 and labels are null, exclude test_labeled)
train_labeled_ids = set(train_labeled['patient_identifier'].values)
test_labeled_ids = set(test_labeled['patient_identifier'].values)

unlabeled_data = df[
    (df['test_set'] == 0) & 
    (df['has_cancer'].isna()) & 
    (df['has_diabetes'].isna()) &
    (~df['patient_identifier'].isin(train_labeled_ids)) &
    (~df['patient_identifier'].isin(test_labeled_ids))
].copy()


if len(unlabeled_data) > 0:
    # Extract unlabeled embeddings
    X_unlabeled = np.vstack(unlabeled_data['embeddings'].values)
    
    # Get probability predictions
    prob_diabetes = clf_diabetes_balanced.predict_proba(X_unlabeled)[:, 1]
    prob_cancer = clf_cancer_balanced.predict_proba(X_unlabeled)[:, 1]
    
    # Apply 60% confidence threshold
    high_conf_diabetes = prob_diabetes >= 0.6
    high_conf_cancer = prob_cancer >= 0.6
    high_conf_neither = (prob_diabetes < 0.2) & (prob_cancer < 0.2)
    
    high_conf_mask = high_conf_diabetes | high_conf_cancer | high_conf_neither
    
    
    if high_conf_mask.sum() > 0:
        ml_synthetic = unlabeled_data[high_conf_mask].copy()
        ml_synthetic['has_diabetes'] = (prob_diabetes[high_conf_mask] >= 0.6).astype(int)
        ml_synthetic['has_cancer'] = (prob_cancer[high_conf_mask] >= 0.6).astype(int)
        ml_synthetic['combined_label'] = ml_synthetic.apply(get_combined_label, axis=1)
        ml_synthetic['source'] = 'ml_synthetic'
    
    else:
        ml_synthetic = pd.DataFrame()
else:
    ml_synthetic = pd.DataFrame()

# ============================================
# STEP 4: PHASE 2 - REGEX-BASED SYNTHETIC LABELING
# ============================================

# Define regex patterns
diabetes_patterns = (
    r'\bT2DM\b|\bT1DM\b|'
    r'\btype\s*[12]\s*diabetes(?:\s*mellitus)?\b|'
    r'\bdiabetes\s*mellitus\b|'
    r'\bhistory\s*of\s*diabetes\b|'
    r'\bdiabetic\b|'
    r'\binsulin\s*(?:aspart|therapy|dependent|glargine|lispro)\b|'
    r'\bmetformin\b|\bglipizide\b|\bglyburide\b'
)

cancer_patterns = (
    r'\bcancer\b|\bcarcinoma\b|\bsarcoma\b|\bmelanoma\b|'
    r'\blymphoma\b|\bleukemia\b|\bmalignancy\b|\bmalignant\b|'
    r'\bneoplasm\b|\bchemotherapy\b|\boncolog(?:y|ist)\b|'
    r'\bmetastatic\b|\btumor\b|\bMDS\s*with\s*excess\s*blasts\b'
)

# Get remaining unlabeled data (exclude Phase 1 ML labels)
if len(ml_synthetic) > 0:
    ml_synthetic_indices = set(ml_synthetic.index)
    remaining_after_ml = unlabeled_data[~unlabeled_data.index.isin(ml_synthetic_indices)].copy()
else:
    remaining_after_ml = unlabeled_data.copy()


if len(remaining_after_ml) > 0:
    # Apply regex
    remaining_after_ml['has_diabetes_regex'] = remaining_after_ml['text'].str.contains(
        diabetes_patterns, case=False, regex=True, na=False
    ).astype(int)
    
    remaining_after_ml['has_cancer_regex'] = remaining_after_ml['text'].str.contains(
        cancer_patterns, case=False, regex=True, na=False
    ).astype(int)
    
    # Create regex synthetic labels
    regex_synthetic = remaining_after_ml[
        (remaining_after_ml['has_diabetes_regex'] == 1) | 
        (remaining_after_ml['has_cancer_regex'] == 1)
    ].copy()
    
    regex_synthetic['has_diabetes'] = regex_synthetic['has_diabetes_regex']
    regex_synthetic['has_cancer'] = regex_synthetic['has_cancer_regex']
    regex_synthetic['combined_label'] = regex_synthetic.apply(get_combined_label, axis=1)
    regex_synthetic['source'] = 'regex'
    
    print(f"Regex synthetic labels: {len(regex_synthetic)}")
    print(f"Distribution: {regex_synthetic['combined_label'].value_counts().to_dict()}")
else:
    regex_synthetic = pd.DataFrame()

# ============================================
# STEP 5: PHASE 3 - ADD 'NEITHER' SYNTHETIC LABELS
# ============================================

# Get remaining unlabeled data (exclude Phase 1 & 2)
used_indices = set()
if len(ml_synthetic) > 0:
    used_indices.update(ml_synthetic.index)
if len(regex_synthetic) > 0:
    used_indices.update(regex_synthetic.index)

if len(unlabeled_data) > 0:
    remaining_after_phase2 = unlabeled_data[~unlabeled_data.index.isin(used_indices)].copy()
    
    
    if len(remaining_after_phase2) > 0:
        X_remaining_neither = np.vstack(remaining_after_phase2['embeddings'].values)
        
        # Predict with baseline models
        prob_diabetes_neither = clf_diabetes_balanced.predict_proba(X_remaining_neither)[:, 1]
        prob_cancer_neither = clf_cancer_balanced.predict_proba(X_remaining_neither)[:, 1]
        
        # RELAXED threshold: both diseases <30%
        neither_mask = (prob_diabetes_neither < 0.3) & (prob_cancer_neither < 0.3)
        
        
        if neither_mask.sum() > 0:
            # Take a sample to balance (target ~200)
            target_neither_count = min(neither_mask.sum(), 200)
            
            ml_synthetic_neither = remaining_after_phase2[neither_mask].sample(
                n=target_neither_count,
                random_state=42
            ).copy()
            
            ml_synthetic_neither['has_diabetes'] = 0
            ml_synthetic_neither['has_cancer'] = 0
            ml_synthetic_neither['combined_label'] = 'Neither'
            ml_synthetic_neither['source'] = 'ml_neither'
            
        else:
            ml_synthetic_neither = pd.DataFrame()
    else:
        ml_synthetic_neither = pd.DataFrame()
else:
    ml_synthetic_neither = pd.DataFrame()

# ============================================
# STEP 6: COMBINE ALL SYNTHETIC DATA
# ============================================

# Add source to train_labeled
train_labeled_copy = train_labeled.copy()
train_labeled_copy['source'] = 'labeled'

# Combine all data
all_dfs = [train_labeled_copy]
if len(ml_synthetic) > 0:
    all_dfs.append(ml_synthetic)
if len(regex_synthetic) > 0:
    all_dfs.append(regex_synthetic)
if len(ml_synthetic_neither) > 0:
    all_dfs.append(ml_synthetic_neither)

final_expanded_balanced = pd.concat(all_dfs, ignore_index=True)


# ============================================
# STEP 7: PREPARE TRAINING DATA
# ============================================

print("\n" + "="*80)
print("STEP 7: PREPARING TRAINING DATA")
print("="*80)

# Baseline data = train_labeled only
X_train_baseline = np.vstack(train_labeled['embeddings'].values)
y_train_baseline = train_labeled['combined_label'].values

# Semi-supervised data = train_labeled + all synthetic
synthetic_only = final_expanded_balanced[final_expanded_balanced['source'] != 'labeled'].copy()

X_train_semisup = np.vstack(final_expanded_balanced['embeddings'].values)
y_train_semisup = final_expanded_balanced['combined_label'].values

print(f"Baseline: {len(y_train_baseline)} examples")
print(f"  Distribution: {pd.Series(y_train_baseline).value_counts().to_dict()}")
print(f"\nSemi-supervised: {len(y_train_semisup)} examples ({len(train_labeled)} labeled + {len(synthetic_only)} synthetic)")
print(f"  Distribution: {pd.Series(y_train_semisup).value_counts().to_dict()}")

# ============================================
# STEP 8: PREPARE TEST DATA
# ============================================

X_test = np.vstack(test_labeled['embeddings'].values)
y_test = test_labeled['combined_label'].values
texts_test = test_labeled['text'].tolist()

print(f"Test set: {len(y_test)} examples")
print(f"Distribution: {pd.Series(y_test).value_counts().to_dict()}")

# ============================================
# STEP 9: TRAIN BASELINE AND SEMI-SUPERVISED MODELS
# ============================================


# Train baseline model (on labeled data only)
clf_baseline_final = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42)
clf_baseline_final.fit(X_train_baseline, y_train_baseline)
print("✓ Baseline model trained")

# Train semi-supervised model (on labeled + synthetic)
clf_semisup_final = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42)
clf_semisup_final.fit(X_train_semisup, y_train_semisup)
print("✓ Semi-supervised model trained")

# Generate predictions and probabilities
y_pred_baseline_final = clf_baseline_final.predict(X_test)
y_pred_baseline_proba = clf_baseline_final.predict_proba(X_test)

y_pred_semisup_final = clf_semisup_final.predict(X_test)
y_pred_semisup_proba = clf_semisup_final.predict_proba(X_test)

# Get class order
classes = clf_baseline_final.classes_

# Store accuracies
baseline_acc_final = accuracy_score(y_test, y_pred_baseline_final)
semisup_acc_final = accuracy_score(y_test, y_pred_semisup_final)

# Rename for compatibility
y_test_final = y_test


# ============================================
# STEP 10: ROUTING ENSEMBLE PREDICTIONS
# ============================================

print("\n" + "="*80)
print("STEP 10: THEORY-DRIVEN ROUTING ENSEMBLE")
print("="*80)

print("\nENSEMBLE STRATEGY (PRE-SPECIFIED):")
print("="*80)
print("JUSTIFICATION: Each model has theoretical strengths based on training:")
print("")
print("1. REGEX (Highest Priority)")
print("   Rationale: High precision when explicit medical keywords present")
print("   Use: Override ML when diabetes/cancer keywords detected -Evidence from Weak Supervision")
print("")
print("2. BASELINE for 'Neither'")
print(f"   Rationale: Trained on balanced data ({(y_train_baseline == 'Neither').sum()}/{len(y_train_baseline)} = {(y_train_baseline == 'Neither').sum()/len(y_train_baseline):.0%} 'Neither')")
print("   Use: When baseline predicts 'Neither', it has seen balanced examples")
print("")
print("3. SEMI-SUPERVISED for Positive Cases")
print(f"   Rationale: {len(y_train_semisup)/len(y_train_baseline):.0f}x more training data for disease patterns")
print("   Use: When predicting Cancer/Diabetes/Both")
print("")
print("4. AGREEMENT")
print("   Rationale: High confidence when both models agree")
print("   Use: Average probabilities when predictions match")
print("")
print("5. WEIGHTED VOTING (Last Resort)")
print("   Rationale: Balance model strengths when they disagree")
print("   Weights: Baseline=1.5x for 'Neither', Semi-sup=2.0x for positive")
print("="*80)

# Create ensemble predictions with PRE-SPECIFIED logic
y_pred_ensemble_final = []
decision_log_final = []

for i in range(len(y_test_final)):
    # Get ML predictions and probabilities
    baseline_pred = y_pred_baseline_final[i]
    semisup_pred = y_pred_semisup_final[i]
    baseline_probs = dict(zip(classes, y_pred_baseline_proba[i]))
    semisup_probs = dict(zip(classes, y_pred_semisup_proba[i]))
    
    # Get patient text and check regex
    patient_text = test_labeled.iloc[i]['text']
    has_diabetes_kw = bool(re.search(diabetes_patterns, patient_text, re.IGNORECASE))
    has_cancer_kw = bool(re.search(cancer_patterns, patient_text, re.IGNORECASE))
    
    # TIER 1: REGEX OVERRIDE
    if has_diabetes_kw and has_cancer_kw:
        prediction = 'Both'
        reason = "REGEX: Both keywords found"
        confidence = 1.0
        
    elif has_diabetes_kw:
        prediction = 'Diabetes Only'
        reason = "REGEX: Diabetes keyword found"
        confidence = 1.0
        
    elif has_cancer_kw:
        prediction = 'Cancer Only'
        reason = "REGEX: Cancer keyword found"
        confidence = 1.0
    
    # TIER 2: MODEL AGREEMENT
    elif baseline_pred == semisup_pred:
        prediction = baseline_pred
        avg_conf = (baseline_probs[baseline_pred] + semisup_probs[semisup_pred]) / 2
        reason = f"AGREEMENT: Both predict {prediction}"
        confidence = avg_conf
    
    # TIER 3: USE MODEL STRENGTHS WHEN DISAGREEING
    elif baseline_pred == 'Neither':
        # Baseline saw balanced 'Neither' examples
        # Semi-supervised saw imbalanced 'Neither'
        # Therefore: Trust baseline for 'Neither' predictions
        prediction = 'Neither'
        reason = "BASELINE strength: Trained on balanced 'Neither' data"
        confidence = baseline_probs['Neither']
        
    elif semisup_pred in ['Cancer Only', 'Diabetes Only', 'Both']:
        # Semi-supervised has more positive case data
        # Therefore: Trust semi-supervised for disease predictions
        prediction = semisup_pred
        reason = "SEMI-SUP strength: More training data for diseases"
        confidence = semisup_probs[semisup_pred]
    
    # TIER 4: WEIGHTED VOTING (fallback)
    else:
        votes = {}
        baseline_weight = 1.5 if baseline_pred == 'Neither' else 1.0
        votes[baseline_pred] = baseline_probs[baseline_pred] * baseline_weight
        
        semisup_weight = 2.0 if semisup_pred != 'Neither' else 1.0
        votes[semisup_pred] = semisup_probs[semisup_pred] * semisup_weight
        
        prediction = max(votes, key=votes.get)
        confidence = votes[prediction] / (baseline_weight + semisup_weight)
        reason = f"WEIGHTED VOTE: {prediction} wins"
    
    y_pred_ensemble_final.append(prediction)
    decision_log_final.append({
        'case_num': i + 1,
        'true': y_test_final[i],
        'prediction': prediction,
        'reason': reason,
        'confidence': confidence,
        'baseline': baseline_pred,
        'baseline_conf': baseline_probs[baseline_pred],
        'semisup': semisup_pred,
        'semisup_conf': semisup_probs[semisup_pred],
        'has_diabetes_kw': has_diabetes_kw,
        'has_cancer_kw': has_cancer_kw
    })

y_pred_ensemble_final = np.array(y_pred_ensemble_final)

# ============================================
# STEP 11: EVALUATION
# ============================================

print("\n" + "="*80)
print("THEORY-DRIVEN ENSEMBLE PERFORMANCE")
print("="*80)
print(classification_report(y_test_final, y_pred_ensemble_final, zero_division=0))

ensemble_final_acc = accuracy_score(y_test_final, y_pred_ensemble_final)

# Show representative examples
# print("\n" + "="*80)
# print("REPRESENTATIVE DECISION EXAMPLES")
# print("="*80)

# decision_types = {}
# for log in decision_log_final:
#     decision_type = log['reason'].split(':')[0]
#     if decision_type not in decision_types:
#         decision_types[decision_type] = []
#     decision_types[decision_type].append(log)

# for decision_type in ['REGEX', 'AGREEMENT', 'BASELINE strength', 'SEMI-SUP strength', 'WEIGHTED VOTE']:
#     if decision_type in decision_types:
#         examples = decision_types[decision_type]
#         correct_ex = next((e for e in examples if e['true'] == e['prediction']), None)
#         incorrect_ex = next((e for e in examples if e['true'] != e['prediction']), None)
        
#         print(f"\n{'─'*60}")
#         print(f"DECISION TYPE: {decision_type}")
#         print(f"{'─'*60}")
        
#         if correct_ex:
#             log = correct_ex
#             print(f"\nExample {log['case_num']} (Correct):")
#             print(f"  True: {log['true']} | Predicted: {log['prediction']} [✓]")
#             print(f"  Reason: {log['reason']}")
#             print(f"  Baseline={log['baseline']} ({log['baseline_conf']:.2f}), Semi-sup={log['semisup']} ({log['semisup_conf']:.2f})")
        
#         if incorrect_ex:
#             log = incorrect_ex
#             print(f"\nExample {log['case_num']} (Incorrect):")
#             print(f"  True: {log['true']} | Predicted: {log['prediction']} [✗]")
#             print(f"  Reason: {log['reason']}")
#             print(f"  Baseline={log['baseline']} ({log['baseline_conf']:.2f}), Semi-sup={log['semisup']} ({log['semisup_conf']:.2f})")

# Final comparison
print("\n" + "="*80)
print("FINAL MODEL COMPARISON")
print("="*80)

comparison_final = []
for cls in ['Neither', 'Cancer Only', 'Diabetes Only', 'Both']:
    true_count = (y_test_final == cls).sum()
    if true_count > 0:
        baseline_recall = ((y_test_final == cls) & (y_pred_baseline_final == cls)).sum() / true_count
        semisup_recall = ((y_test_final == cls) & (y_pred_semisup_final == cls)).sum() / true_count
        ensemble_recall = ((y_test_final == cls) & (y_pred_ensemble_final == cls)).sum() / true_count
        
        comparison_final.append({
            'Class': cls,
            'Count': true_count,
            'Baseline': f"{baseline_recall:.0%}",
            'Semi-Sup': f"{semisup_recall:.0%}",
            'Ensemble': f"{ensemble_recall:.0%}"
        })

print("\n", pd.DataFrame(comparison_final).to_string(index=False))

print(f"\n{'='*80}")
print("OVERALL ACCURACY")
print(f"{'='*80}")
print(f"  Baseline:               {baseline_acc_final:.1%}")
print(f"  Semi-Supervised:        {semisup_acc_final:.1%}")
print(f"  Theory-Driven Ensemble: {ensemble_final_acc:.1%}")



for cls in ['Diabetes Only', 'Cancer Only', 'Both', 'Neither']:
    if (y_test_final == cls).sum() > 0:
        correct = ((y_test_final == cls) & (y_pred_ensemble_final == cls)).sum()
        total = (y_test_final == cls).sum()
        print(f"✓ {cls} recall: {correct}/{total} = {correct/total:.0%}")

print(f"\n{'='*80}")
print("✓ COMPLETE ENSEMBLE PIPELINE FINISHED")
print(f"{'='*80}")

FINAL ENSEMBLE: COMPLETE STANDALONE PIPELINE
✓ Binary classifiers trained on train_labeled
Regex synthetic labels: 762
Distribution: {'Cancer Only': 639, 'Diabetes Only': 90, 'Both': 33}

STEP 7: PREPARING TRAINING DATA
Baseline: 36 examples
  Distribution: {'Neither': 21, 'Cancer Only': 13, 'Diabetes Only': 1, 'Both': 1}

Semi-supervised: 903 examples (36 labeled + 867 synthetic)
  Distribution: {'Cancer Only': 757, 'Diabetes Only': 91, 'Both': 34, 'Neither': 21}
Test set: 14 examples
Distribution: {'Neither': 6, 'Cancer Only': 5, 'Diabetes Only': 2, 'Both': 1}
✓ Baseline model trained
✓ Semi-supervised model trained

STEP 10: THEORY-DRIVEN ROUTING ENSEMBLE

ENSEMBLE STRATEGY (PRE-SPECIFIED):
JUSTIFICATION: Each model has theoretical strengths based on training:

1. REGEX (Highest Priority)
   Rationale: High precision when explicit medical keywords present
   Use: Override ML when diabetes/cancer keywords detected -Evidence from Weak Supervision

2. BASELINE for 'Neither'
   Rational

# Experimental Approach

In [112]:
df_test = df[df['test_set'] == 1].copy()

df_labeled = df[
    (df['test_set'] == 0) & 
    (df['has_cancer'].notna()) & 
    (df['has_diabetes'].notna())
].copy()

df_unlabeled = df[
    (df['test_set'] == 0) & 
    (df['has_cancer'].isna() | df['has_diabetes'].isna())
].copy()

In [113]:
from sklearn.model_selection import train_test_split

# Create a combined label column for stratification
def create_combined_label(row):
    has_cancer = row['has_cancer'] == 1.0
    has_diabetes = row['has_diabetes'] == 1.0
    
    if has_cancer and has_diabetes:
        return 'Both'
    elif has_cancer:
        return 'Cancer Only'
    elif has_diabetes:
        return 'Diabetes Only'
    else:
        return 'Neither'

df_labeled['combined_label'] = df_labeled.apply(create_combined_label, axis=1)


# Stratified split (80/20 or 70/30 - adjust test_size as needed)
df_train, df_validation = train_test_split(
    df_labeled,
    test_size=0.2,  # 20% for validation, adjust if you prefer 0.3 (30%)
    stratify=df_labeled['combined_label'],
    random_state=42
)

# Verify stratification worked
print("Training Set Distribution")
print("=" * 50)
print(df_train['combined_label'].value_counts())
print(f"\nTotal: {len(df_train)}")
print()

print("Validation Set Distribution")
print("=" * 50)
print(df_validation['combined_label'].value_counts())
print(f"\nTotal: {len(df_validation)}")
print()


Training Set Distribution
combined_label
Neither          22
Cancer Only      14
Both              2
Diabetes Only     2
Name: count, dtype: int64

Total: 40

Validation Set Distribution
combined_label
Neither          5
Cancer Only      4
Diabetes Only    1
Name: count, dtype: int64

Total: 10



In [ ]:
import os
from openai import OpenAI
from langchain_astradb import AstraDBVectorStore
from langchain_openai import OpenAIEmbeddings
from astrapy import DataAPIClient

# Set environment variables
os.environ['OPENAI_API_KEY'] = ''
os.environ['ASTRA_DB_APPLICATION_TOKEN'] = ''
os.environ['ASTRA_DB_API_ENDPOINT'] = ''

# Initialize OpenAI
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# Initialize Astra DB
astra_client = DataAPIClient(os.environ['ASTRA_DB_APPLICATION_TOKEN'])
db = astra_client.get_database_by_api_endpoint(os.environ['ASTRA_DB_API_ENDPOINT'])

collection_name = "patient_embeddings"
if collection_name in db.list_collection_names():
    db.drop_collection(collection_name)
    print(f"Dropped existing collection: {collection_name}")

# Initialize LangChain vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = AstraDBVectorStore(
    collection_name=collection_name,
    embedding=embeddings,
    token=os.environ['ASTRA_DB_APPLICATION_TOKEN'],
    api_endpoint=os.environ['ASTRA_DB_API_ENDPOINT'],
)
print(f"Created vector store: {collection_name}")

# Prepare data for batch insertion
texts = df_train['text'].tolist()
metadatas = [
    {
        "patient_identifier": str(row['patient_identifier']),
        "has_cancer": float(row['has_cancer']),
        "has_diabetes": float(row['has_diabetes']),
        "combined_label": row['combined_label']
    }
    for _, row in df_train.iterrows()
]

# Add texts with metadata (embeddings generated automatically)
document_ids = vector_store.add_texts(texts=texts, metadatas=metadatas)
print(f"Stored {len(document_ids)} embeddings in Astra DB")

Dropped existing collection: patient_embeddings
Created vector store: patient_embeddings
Stored 40 embeddings in Astra DB


In [115]:
def similarity_search_agent(query_text, top_k=5):
    """
    Agent 1: Similarity Search
    Retrieves similar patient cases from the vector store
    """
    # Perform similarity search
    results = vector_store.similarity_search_with_score(
        query=query_text,
        k=top_k
    )
    
    # Extract and format results
    retrieved_cases = []
    for doc, score in results:
        case = {
            'text': doc.page_content,
            'has_cancer': doc.metadata['has_cancer'],
            'has_diabetes': doc.metadata['has_diabetes'],
            'combined_label': doc.metadata['combined_label'],
            'patient_id': doc.metadata['patient_identifier'],
            'similarity_score': score
        }
        retrieved_cases.append(case)
    
    return retrieved_cases

# Test the agent with a sample from validation set
test_patient = df_validation.iloc[0]
print(f"Query Patient ID: {test_patient['patient_identifier']}")
print(f"True Label: {test_patient['combined_label']}")
print(f"\nQuery Text (first 300 chars):\n{test_patient['text'][:300]}...")
print("\n" + "="*80)
print("SIMILAR CASES RETRIEVED:")
print("="*80)

similar_cases = similarity_search_agent(test_patient['text'], top_k=5)

for i, case in enumerate(similar_cases, 1):
    print(f"\n{i}. Patient ID: {case['patient_id']} (Similarity: {case['similarity_score']:.3f})")
    print(f"   Label: {case['combined_label']}")
    print(f"   Has Cancer: {case['has_cancer']}, Has Diabetes: {case['has_diabetes']}")
    print(f"   Text: {case['text'][:200]}...")

Query Patient ID: 1047
True Label: Cancer Only

Query Text (first 300 chars):
Discharge Summary

Patient Name: [REDACTED]
Gender: Male
Age: 62 years
Medical Record Number: [REDACTED]

Hospital Course:
The patient was admitted to the hospital for the treatment of bullous hemorrhagic dermatosis. He had a history of morbid obesity, coronary artery disease, lipodermatosclerosis o...

SIMILAR CASES RETRIEVED:

1. Patient ID: 2563 (Similarity: 0.819)
   Label: Neither
   Has Cancer: 0, Has Diabetes: 0
   Text: Discharge Summary:
Patient name: [REDACTED]
Sex: Female
Age: 70 years
Admission date: [REDACTED]
Discharge date: [REDACTED]

Brief Hospital Course:
The patient was admitted with lethargy, anorexia, na...

2. Patient ID: 2183 (Similarity: 0.807)
   Label: Cancer Only
   Has Cancer: 1, Has Diabetes: 0
   Text: DISCHARGE SUMMARY

Patient Name: [REDACTED]

Medical Record Number: [REDACTED]

Admission Date: [REDACTED]

Discharge Date: [REDACTED]

Admitting Diagnosis: Adenocarcinoma suspecte

In [116]:
similarity_search_agent("Patient with history of type 2 diabetes and recent diagnosis of carcinoma.", top_k=3)

[{'text': 'Hospital Course:\nPatient 2, a 32-year-old male with a history of CDH1 mutation and HDGC, presented for prophylactic total gastrectomy. The procedure was completed without complications, and the patient was discharged home on post-operative day 7.\n\nAdmission:\nThe patient returned to the hospital 5 days later with symptoms of diffuse abdominal pain, dark-colored emesis, and no bowel movements for 2 days. Abdominal CT scan revealed a dilated, gas-filled small bowel loop. However, subsequent tests were unremarkable except for elevated amylase and lipase levels, which suggested pancreatitis as a more likely source for his pain. The patient was treated for pancreatitis and discharged without further complications.\n\nDischarge Condition:\nAt the time of this report, patient 2 is well with no evidence of disease.\n\nSummary:\nPatient 2 underwent prophylactic total gastrectomy due to the presence of the CDH1 mutation and HDGC. The pathology report showed microscopic foci of sign

In [117]:
unlabeled_data

,patient_identifier,text,has_cancer,has_diabetes,test_set,embeddings
50,1013,Discharge Summary:\n\nPatient Name: [REDACTED]...,NaN,NaN,0,"[0.030318165197968483, 0.027921970933675766, 0..."
51,2988,Hospital Course:\nA 5-year-old male patient on...,NaN,NaN,0,"[0.03922712057828903, -0.00038238795241340995,..."
52,412,Discharge Summary:\n\nPatient: 42-year-old mal...,NaN,NaN,0,"[0.04739338904619217, -0.006408666260540485, 0..."
53,1056,Discharge Summary:\n\nPatient Overview:\n• 53-...,NaN,NaN,0,"[0.010367999784648418, 0.00608861306682229, 0...."
54,302,Discharge Summary:\n\nPatient Name: [REDACTED]...,NaN,NaN,0,"[-0.006083865649998188, 0.023505395278334618, ..."
...,...,...,...,...,...,...
1995,699,Discharge Summary:\n\nPatient: 50-year-old mal...,NaN,NaN,0,"[0.06719277799129486, 0.013694754801690578, 0...."
1996,2334,Discharge Summary:\n\nPatient Name: Unavailabl...,NaN,NaN,0,"[0.02454804629087448, -0.021654067561030388, -..."
1997,1868,Discharge Summary:\n\nPatient was a 12-year-ol...,NaN,NaN,0,"[0.01788659580051899, 0.007300006691366434, -0..."
1998,449,Discharge Summary:\n\nPatient Name: [redacted]...,NaN,NaN,0,"[0.03324780613183975, 0.036069124937057495, 0...."


In [118]:
HEALTHCARE_CLASSIFICATION_PROMPT = """You are a medical record classification expert specializing in identifying CURRENT cancer and diabetes diagnoses from hospital discharge summaries.

Your task is to classify whether a patient CURRENTLY has:
- Cancer (any active malignancy)
- Diabetes (Type 1, Type 2, or gestational diabetes mellitus) 
- Both conditions
- Neither condition

CRITICAL: You are looking for CURRENT Diabetes, and ANY form of Cancer

=================================================================================
DIABETES DIAGNOSTIC CRITERIA (Source: American Diabetes Association 2024-2025)
=================================================================================

DEFINITIVE DIABETES INDICATORS:
1. Explicit diagnosis statements:
   - "diabetes mellitus", "type 1 diabetes", "type 2 diabetes", "T1DM", "T2DM"
   - "gestational diabetes mellitus" (GDM)
   
2. Laboratory criteria (any ONE of the following):
   - Fasting plasma glucose (FPG) ≥126 mg/dL (7.0 mmol/L)
   - 2-hour plasma glucose ≥200 mg/dL (11.1 mmol/L) during 75-g oral glucose tolerance test
   - HbA1c ≥6.5%
   - Random plasma glucose ≥200 mg/dL (11.1 mmol/L) WITH classic hyperglycemic symptoms

3. Classic hyperglycemic symptoms (when combined with elevated glucose):
   - Polyuria (increased urination) + Polydipsia (increased thirst)
   - Unexplained weight loss
   - These symptoms ALONE are insufficient without confirmed diagnosis or lab values

4. Diabetes-specific treatments:
   - Insulin therapy (aspart, glargine, lispro, NPH, etc.)
   - Oral hypoglycemics: metformin, sulfonylureas, SGLT2 inhibitors, DPP-4 inhibitors
   
5. Diabetes complications (indicate existing diabetes):
   - Diabetic ketoacidosis (DKA)
   - Diabetic retinopathy, nephropathy, or neuropathy
   - Hyperosmolar hyperglycemic state

DO NOT classify as diabetes:
- "Prediabetes" or "impaired fasting glucose" (IFG 100-125 mg/dL)
- "Impaired glucose tolerance" (IGT)
- Transient hyperglycemia (stress, steroids, acute illness)
- "Polyuria and polydipsia" without confirmed diagnosis
- "Rule out diabetes" or "screening for diabetes"

=================================================================================
CANCER DIAGNOSTIC CRITERIA (Source: MSD Manual, NCI, Multiple Medical Sources)
=================================================================================

DEFINITIVE CANCER INDICATORS:
1. Explicit malignancy diagnoses:
   - Any cancer type: carcinoma, adenocarcinoma, lymphoma, leukemia, sarcoma, melanoma
   - Specific examples: glioblastoma, myeloma, mesothelioma, neuroblastoma
   - Staging information: TNM staging, Stage 0-IV, AJCC staging

2. Histopathologic confirmation (gold standard):
   - "Biopsy confirmed malignancy"
   - "Pathology report shows [cancer type]"
   - "Histologic diagnosis of [cancer]"

3. Cancer-specific treatments:
   - Chemotherapy for malignancy
   - Radiation therapy for cancer
   - Surgical resection of malignant tumor
   - Immunotherapy, targeted therapy for cancer

4. Metastatic disease:
   - "Metastases to [organ]"
   - "Stage IV disease"
   - "Distant spread"

DO NOT classify as cancer:
- Benign tumors, cysts, polyps
- "Mass" or "lesion" without biopsy confirmation
- "Suspicious for malignancy" without confirmation
- "Rule out cancer"
- Precancerous conditions (dysplasia, carcinoma in situ may or may not be cancer depending on context)

=================================================================================
TEMPORAL CONTEXT RULES
=================================================================================

ACTIVE/CURRENT diagnosis = HAS the condition:
- "Patient has diabetes"
- "History of diabetes" (chronic condition, still present)
- "Known diabetic"
- "Patient's diabetes"
- "Active malignancy"
- "Undergoing chemotherapy for [cancer]"

PAST/RESOLVED = DOES NOT HAVE the condition:
- "Previous cancer, now in remission"
- "History of cancer, successfully resected 10 years ago"
- "Past diagnosis of [condition], no longer active"

UNDER INVESTIGATION = DOES NOT HAVE the condition:
- "Rule out cancer"
- "Suspected malignancy"
- "Evaluate for diabetes"

=================================================================================
CLASSIFICATION PROCESS
=================================================================================

1. Read the ENTIRE discharge summary
2. Look for explicit diagnosis statements (primary diagnoses, medical history, problem list)
3. Identify definitive treatments indicating the condition
4. Check for lab values meeting diagnostic criteria
5. Distinguish between current vs. past diagnoses
6. Distinguish between confirmed vs. suspected diagnoses

OUTPUT FORMAT:
Classification: [Cancer Only / Diabetes Only / Both / Neither]
Has Cancer: [0.0 or 1.0]
Has Diabetes: [0.0 or 1.0]
Confidence: [0.0-1.0]
Key Evidence: [Quote 1-2 specific phrases from text that support classification]
Reasoning: [2-3 sentences explaining decision with source-backed criteria]

Now classify this patient discharge summary:

{patient_text}
"""

In [119]:
def llm_classification_agent(patient_text, model="gpt-4o-mini"):
    """
    Agent 2: LLM-based Classification
    Uses strict healthcare context to classify cancer/diabetes
    
    Returns:
        dict with keys: classification, has_cancer, has_diabetes, confidence, key_evidence, reasoning
    """
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    
    prompt = HEALTHCARE_CLASSIFICATION_PROMPT.format(patient_text=patient_text)
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a medical record classification expert."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )
    
    response_text = response.choices[0].message.content
    
    # Parse the response to extract structured data
    result = {
        'raw_response': response_text,
        'classification': None,
        'has_cancer': None,
        'has_diabetes': None,
        'confidence': None,
        'key_evidence': None,
        'reasoning': None
    }
    
    # Parse each field from response
    for line in response_text.split('\n'):
        if line.startswith('Classification:'):
            result['classification'] = line.replace('Classification:', '').strip()
        elif line.startswith('Has Cancer:'):
            result['has_cancer'] = float(line.replace('Has Cancer:', '').strip())
        elif line.startswith('Has Diabetes:'):
            result['has_diabetes'] = float(line.replace('Has Diabetes:', '').strip())
        elif line.startswith('Confidence:'):
            result['confidence'] = float(line.replace('Confidence:', '').strip())
        elif line.startswith('Key Evidence:'):
            result['key_evidence'] = line.replace('Key Evidence:', '').strip()
        elif line.startswith('Reasoning:'):
            result['reasoning'] = line.replace('Reasoning:', '').strip()
    
    return result

# Test the agent on validation samples
print("LLM CLASSIFICATION AGENT - VALIDATION TEST")
print("="*80)

test_results = []

for i in range(min(5, len(df_validation))):
    test_patient = df_validation.iloc[i]
    
    print(f"\n{'='*80}")
    print(f"TEST CASE {i+1}")
    print(f"{'='*80}")
    print(f"Patient ID: {test_patient['patient_identifier']}")
    print(f"True Label: {test_patient['combined_label']}")
    print(f"True Has Cancer: {test_patient['has_cancer']}")
    print(f"True Has Diabetes: {test_patient['has_diabetes']}")
    print(f"\nText Preview (first 400 chars):\n{test_patient['text'][:400]}...")
    
    print(f"\n{'-'*80}")
    print("LLM CLASSIFICATION:")
    print(f"{'-'*80}")
    
    classification = llm_classification_agent(test_patient['text'])
    
    print(f"Classification: {classification['classification']}")
    print(f"Has Cancer: {classification['has_cancer']}")
    print(f"Has Diabetes: {classification['has_diabetes']}")
    print(f"Confidence: {classification['confidence']}")
    print(f"Key Evidence: {classification['key_evidence']}")
    print(f"Reasoning: {classification['reasoning']}")
    
    # Check if correct
    correct_cancer = classification['has_cancer'] == test_patient['has_cancer']
    correct_diabetes = classification['has_diabetes'] == test_patient['has_diabetes']
    correct_overall = correct_cancer and correct_diabetes
    
    print(f"\n✓ CORRECT" if correct_overall else f"\n✗ INCORRECT")
    if not correct_cancer:
        print(f"  Cancer mismatch: predicted {classification['has_cancer']}, actual {test_patient['has_cancer']}")
    if not correct_diabetes:
        print(f"  Diabetes mismatch: predicted {classification['has_diabetes']}, actual {test_patient['has_diabetes']}")
    
    test_results.append({
        'patient_id': test_patient['patient_identifier'],
        'true_label': test_patient['combined_label'],
        'predicted_label': classification['classification'],
        'correct': correct_overall,
        'confidence': classification['confidence']
    })

# Summary
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
correct_count = sum(1 for r in test_results if r['correct'])
print(f"Accuracy: {correct_count}/{len(test_results)} ({correct_count/len(test_results)*100:.1f}%)")
print(f"Average Confidence: {sum(r['confidence'] for r in test_results if r['confidence'])/len([r for r in test_results if r['confidence']]):.2f}")

LLM CLASSIFICATION AGENT - VALIDATION TEST

TEST CASE 1
Patient ID: 1047
True Label: Cancer Only
True Has Cancer: 1.0
True Has Diabetes: 0.0

Text Preview (first 400 chars):
Discharge Summary

Patient Name: [REDACTED]
Gender: Male
Age: 62 years
Medical Record Number: [REDACTED]

Hospital Course:
The patient was admitted to the hospital for the treatment of bullous hemorrhagic dermatosis. He had a history of morbid obesity, coronary artery disease, lipodermatosclerosis of the lower extremities, chronic peripheral venous insufficiency, and prostate cancer (Gleason 4+5) ...

--------------------------------------------------------------------------------
LLM CLASSIFICATION:
--------------------------------------------------------------------------------
Classification: Both
Has Cancer: 1.0
Has Diabetes: 0.0
Confidence: 0.9
Key Evidence: "history of prostate cancer (Gleason 4+5) on long-term androgen deprivation therapy"
Reasoning: The patient has an active malignancy, specifically prostat

In [120]:
FINAL_DECISION_PROMPT = """You are a senior medical record analyst making FINAL classification decisions for cancer and diabetes diagnoses.

You have been provided with:
1. Similar patient cases from a medical database (RETRIEVAL CONTEXT)
2. An initial AI classification with reasoning and confidence (INITIAL CLASSIFICATION)
3. The current patient's discharge summary (PATIENT TEXT)

Your task is to make the FINAL, DEFINITIVE classification by weighing all available evidence.

=================================================================================
DECISION-MAKING FRAMEWORK
=================================================================================

STEP 1: EVALUATE RETRIEVAL CONTEXT
- How many similar cases have cancer? Diabetes? Both? Neither?
- What is the similarity score of the top matches?
- Do the similar cases provide strong evidence for a pattern?

STEP 2: EVALUATE INITIAL CLASSIFICATION
- What was the AI's classification and confidence?
- Is the reasoning sound based on diagnostic criteria?
- Did it cite specific evidence from the text?

STEP 3: EXAMINE PATIENT TEXT DIRECTLY
- What explicit diagnoses are stated?
- What treatments indicate cancer or diabetes?
- Are there lab values meeting diagnostic thresholds?
- Is the condition current/active vs. past/resolved?

STEP 4: MAKE FINAL DECISION
- If retrieval context and initial classification AGREE → High confidence
- If retrieval context and initial classification DISAGREE → Examine patient text carefully to break tie
- If initial classification has LOW confidence (<0.6) → Give more weight to retrieval context
- If similar cases have MIXED labels → Give more weight to direct text analysis

=================================================================================
DIAGNOSTIC CRITERIA REFERENCE
=================================================================================

DIABETES:
- Explicit: "diabetes mellitus", "T1DM", "T2DM", "history of diabetes"
- Labs: FPG ≥126 mg/dL, HbA1c ≥6.5%, random glucose ≥200 mg/dL with symptoms
- Treatments: insulin, metformin, sulfonylureas
- Complications: DKA, diabetic retinopathy/nephropathy/neuropathy
- NOT diabetes: "polyuria/polydipsia" alone, prediabetes, transient hyperglycemia

CANCER:
- Explicit: carcinoma, lymphoma, leukemia, sarcoma, melanoma, with staging
- Confirmation: biopsy-proven malignancy, histopathology
- Treatments: chemotherapy, radiation therapy, surgical resection of malignant tumor
- Metastases: Stage IV, distant spread, metastases to organs
- NOT cancer: benign tumors, "rule out cancer", suspicious but unconfirmed lesions

TEMPORAL CONTEXT:
- "History of [condition]" for chronic diseases = CURRENT diagnosis
- "Previous [condition], now in remission" = NOT current
- "Rule out [condition]" = NOT confirmed

=================================================================================
OUTPUT FORMAT (STRICT - MUST FOLLOW EXACTLY)
=================================================================================

Final Classification: [Cancer Only / Diabetes Only / Both / Neither]
Has Cancer: [0.0 or 1.0]
Has Diabetes: [0.0 or 1.0]
Final Confidence: [0.0-1.0]
Decision Rationale: [3-4 sentences explaining how you weighed the evidence from retrieval, initial classification, and direct text analysis]

=================================================================================
INPUTS FOR CURRENT CASE
=================================================================================

RETRIEVAL CONTEXT (Similar Patient Cases):
{retrieval_context}

INITIAL CLASSIFICATION:
Classification: {initial_classification}
Has Cancer: {initial_has_cancer}
Has Diabetes: {initial_has_diabetes}
Confidence: {initial_confidence}
Reasoning: {initial_reasoning}

PATIENT DISCHARGE SUMMARY:
{patient_text}

Now make your FINAL classification decision:
"""

In [121]:
def final_decision_agent(patient_text, similar_cases, initial_classification_result, model="gpt-4o"):
    """
    Agent 3: Final Decision Maker
    Combines retrieval context + initial classification to make final decision
    
    Args:
        patient_text: Current patient's discharge summary
        similar_cases: List of dicts from similarity_search_agent
        initial_classification_result: Dict from llm_classification_agent
        model: OpenAI model to use (gpt-4o for better reasoning)
    
    Returns:
        dict with keys: final_classification, has_cancer, has_diabetes, final_confidence, decision_rationale
    """
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    
    # Format retrieval context
    retrieval_text = ""
    for i, case in enumerate(similar_cases, 1):
        retrieval_text += f"\nSimilar Case {i} (Similarity: {case['similarity_score']:.3f}):\n"
        retrieval_text += f"  Label: {case['combined_label']}\n"
        retrieval_text += f"  Has Cancer: {case['has_cancer']}, Has Diabetes: {case['has_diabetes']}\n"
        retrieval_text += f"  Text Preview: {case['text'][:200]}...\n"
    
    # Build prompt
    prompt = FINAL_DECISION_PROMPT.format(
        retrieval_context=retrieval_text,
        initial_classification=initial_classification_result['classification'],
        initial_has_cancer=initial_classification_result['has_cancer'],
        initial_has_diabetes=initial_classification_result['has_diabetes'],
        initial_confidence=initial_classification_result['confidence'],
        initial_reasoning=initial_classification_result['reasoning'],
        patient_text=patient_text[:2000]  # Limit text length for context window
    )
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a senior medical record analyst making final classification decisions."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )
    
    response_text = response.choices[0].message.content
    
    # Parse the response
    result = {
        'raw_response': response_text,
        'final_classification': None,
        'has_cancer': None,
        'has_diabetes': None,
        'final_confidence': None,
        'decision_rationale': None
    }
    
    for line in response_text.split('\n'):
        if line.startswith('Final Classification:'):
            result['final_classification'] = line.replace('Final Classification:', '').strip()
        elif line.startswith('Has Cancer:'):
            result['has_cancer'] = float(line.replace('Has Cancer:', '').strip())
        elif line.startswith('Has Diabetes:'):
            result['has_diabetes'] = float(line.replace('Has Diabetes:', '').strip())
        elif line.startswith('Final Confidence:'):
            result['final_confidence'] = float(line.replace('Final Confidence:', '').strip())
        elif line.startswith('Decision Rationale:'):
            result['decision_rationale'] = line.replace('Decision Rationale:', '').strip()
    
    return result

# Full pipeline test
print("COMPLETE PIPELINE TEST - THREE AGENT SYSTEM")
print("="*80)

pipeline_results = []

for i in range(min(5, len(df_validation))):
    test_patient = df_validation.iloc[i]
    
    print(f"\n{'='*80}")
    print(f"PIPELINE TEST CASE {i+1}")
    print(f"{'='*80}")
    print(f"Patient ID: {test_patient['patient_identifier']}")
    print(f"True Label: {test_patient['combined_label']}")
    print(f"True Has Cancer: {test_patient['has_cancer']}")
    print(f"True Has Diabetes: {test_patient['has_diabetes']}")
    
    # AGENT 1: Similarity Search
    print(f"\n{'-'*80}")
    print("AGENT 1: SIMILARITY SEARCH")
    print(f"{'-'*80}")
    similar_cases = similarity_search_agent(test_patient['text'], top_k=5)
    
    # Count labels in similar cases
    cancer_count = sum(1 for c in similar_cases if c['has_cancer'] == 1.0)
    diabetes_count = sum(1 for c in similar_cases if c['has_diabetes'] == 1.0)
    print(f"Similar cases - Cancer: {cancer_count}/5, Diabetes: {diabetes_count}/5")
    for j, case in enumerate(similar_cases[:3], 1):
        print(f"  {j}. {case['combined_label']} (similarity: {case['similarity_score']:.3f})")
    
    # AGENT 2: Initial Classification
    print(f"\n{'-'*80}")
    print("AGENT 2: INITIAL LLM CLASSIFICATION")
    print(f"{'-'*80}")
    initial_classification = llm_classification_agent(test_patient['text'])
    print(f"Classification: {initial_classification['classification']}")
    print(f"Has Cancer: {initial_classification['has_cancer']}")
    print(f"Has Diabetes: {initial_classification['has_diabetes']}")
    print(f"Confidence: {initial_classification['confidence']}")
    print(f"Reasoning: {initial_classification['reasoning']}")
    
    # AGENT 3: Final Decision
    print(f"\n{'-'*80}")
    print("AGENT 3: FINAL DECISION")
    print(f"{'-'*80}")
    final_decision = final_decision_agent(
        test_patient['text'], 
        similar_cases, 
        initial_classification
    )
    print(f"Final Classification: {final_decision['final_classification']}")
    print(f"Has Cancer: {final_decision['has_cancer']}")
    print(f"Has Diabetes: {final_decision['has_diabetes']}")
    print(f"Final Confidence: {final_decision['final_confidence']}")
    print(f"Decision Rationale: {final_decision['decision_rationale']}")
    
    # Evaluation
    correct_cancer = final_decision['has_cancer'] == test_patient['has_cancer']
    correct_diabetes = final_decision['has_diabetes'] == test_patient['has_diabetes']
    correct_overall = correct_cancer and correct_diabetes
    
    print(f"\n{'✓ CORRECT' if correct_overall else '✗ INCORRECT'}")
    if not correct_cancer:
        print(f"  Cancer mismatch: predicted {final_decision['has_cancer']}, actual {test_patient['has_cancer']}")
    if not correct_diabetes:
        print(f"  Diabetes mismatch: predicted {final_decision['has_diabetes']}, actual {test_patient['has_diabetes']}")
    
    pipeline_results.append({
        'patient_id': test_patient['patient_identifier'],
        'true_label': test_patient['combined_label'],
        'initial_classification': initial_classification['classification'],
        'final_classification': final_decision['final_classification'],
        'initial_confidence': initial_classification['confidence'],
        'final_confidence': final_decision['final_confidence'],
        'correct': correct_overall
    })

# Summary
print(f"\n{'='*80}")
print("PIPELINE SUMMARY")
print(f"{'='*80}")
correct_count = sum(1 for r in pipeline_results if r['correct'])
print(f"Final Accuracy: {correct_count}/{len(pipeline_results)} ({correct_count/len(pipeline_results)*100:.1f}%)")
print(f"Average Initial Confidence: {sum(r['initial_confidence'] for r in pipeline_results if r['initial_confidence'])/len([r for r in pipeline_results if r['initial_confidence']]):.2f}")
print(f"Average Final Confidence: {sum(r['final_confidence'] for r in pipeline_results if r['final_confidence'])/len([r for r in pipeline_results if r['final_confidence']]):.2f}")

# Show cases where classification changed
changed_cases = [r for r in pipeline_results if r['initial_classification'] != r['final_classification']]
if changed_cases:
    print(f"\nCases where final decision differed from initial: {len(changed_cases)}")
    for case in changed_cases:
        print(f"  Patient {case['patient_id']}: {case['initial_classification']} → {case['final_classification']}")

COMPLETE PIPELINE TEST - THREE AGENT SYSTEM

PIPELINE TEST CASE 1
Patient ID: 1047
True Label: Cancer Only
True Has Cancer: 1.0
True Has Diabetes: 0.0

--------------------------------------------------------------------------------
AGENT 1: SIMILARITY SEARCH
--------------------------------------------------------------------------------
Similar cases - Cancer: 2/5, Diabetes: 0/5
  1. Neither (similarity: 0.819)
  2. Cancer Only (similarity: 0.807)
  3. Cancer Only (similarity: 0.804)

--------------------------------------------------------------------------------
AGENT 2: INITIAL LLM CLASSIFICATION
--------------------------------------------------------------------------------
Classification: Both
Has Cancer: 1.0
Has Diabetes: 0.0
Confidence: 0.9
Reasoning: The patient has an active diagnosis of prostate cancer, indicated by the explicit mention of "prostate cancer (Gleason 4+5) on long-term androgen deprivation therapy." However, there is no mention of diabetes or any indicators t

In [122]:
# Full pipeline test on entire validation set
print("COMPLETE PIPELINE TEST - THREE AGENT SYSTEM")
print("="*80)

pipeline_results = []

for i in range(len(df_validation)):
    test_patient = df_validation.iloc[i]
    
    print(f"\nProcessing case {i+1}/{len(df_validation)}...", end='\r')
    
    # AGENT 1: Similarity Search
    similar_cases = similarity_search_agent(test_patient['text'], top_k=5)
    
    # AGENT 2: Initial Classification
    initial_classification = llm_classification_agent(test_patient['text'])
    
    # AGENT 3: Final Decision
    final_decision = final_decision_agent(
        test_patient['text'], 
        similar_cases, 
        initial_classification
    )
    
    # Evaluation
    correct_cancer = final_decision['has_cancer'] == test_patient['has_cancer']
    correct_diabetes = final_decision['has_diabetes'] == test_patient['has_diabetes']
    correct_overall = correct_cancer and correct_diabetes
    
    pipeline_results.append({
        'patient_id': test_patient['patient_identifier'],
        'true_label': test_patient['combined_label'],
        'true_has_cancer': test_patient['has_cancer'],
        'true_has_diabetes': test_patient['has_diabetes'],
        'initial_classification': initial_classification['classification'],
        'final_classification': final_decision['final_classification'],
        'pred_has_cancer': final_decision['has_cancer'],
        'pred_has_diabetes': final_decision['has_diabetes'],
        'initial_confidence': initial_classification['confidence'],
        'final_confidence': final_decision['final_confidence'],
        'correct': correct_overall,
        'correct_cancer': correct_cancer,
        'correct_diabetes': correct_diabetes
    })

print("\n" + "="*80)
print("FINAL EVALUATION STATISTICS")
print("="*80)

# Overall accuracy
overall_accuracy = sum(r['correct'] for r in pipeline_results) / len(pipeline_results)
print(f"\nOverall Accuracy: {overall_accuracy:.1%} ({sum(r['correct'] for r in pipeline_results)}/{len(pipeline_results)})")

# Cancer detection metrics
cancer_accuracy = sum(r['correct_cancer'] for r in pipeline_results) / len(pipeline_results)
print(f"\nCancer Detection Accuracy: {cancer_accuracy:.1%}")

y_true_cancer = [r['true_has_cancer'] for r in pipeline_results]
y_pred_cancer = [r['pred_has_cancer'] for r in pipeline_results]
print("\nCancer Classification Report:")
print(classification_report(y_true_cancer, y_pred_cancer, target_names=['No Cancer', 'Cancer']))

# Diabetes detection metrics
diabetes_accuracy = sum(r['correct_diabetes'] for r in pipeline_results) / len(pipeline_results)
print(f"\nDiabetes Detection Accuracy: {diabetes_accuracy:.1%}")

y_true_diabetes = [r['true_has_diabetes'] for r in pipeline_results]
y_pred_diabetes = [r['pred_has_diabetes'] for r in pipeline_results]
print("\nDiabetes Classification Report:")
print(classification_report(y_true_diabetes, y_pred_diabetes, target_names=['No Diabetes', 'Diabetes']))

# Combined label accuracy
y_true_combined = [r['true_label'] for r in pipeline_results]
y_pred_combined = [r['final_classification'] for r in pipeline_results]
print("\nCombined Label Classification Report:")
print(classification_report(y_true_combined, y_pred_combined))

# Confidence statistics
print(f"\nConfidence Statistics:")
print(f"  Average Initial Confidence: {sum(r['initial_confidence'] for r in pipeline_results if r['initial_confidence'])/len([r for r in pipeline_results if r['initial_confidence']]):.3f}")
print(f"  Average Final Confidence: {sum(r['final_confidence'] for r in pipeline_results if r['final_confidence'])/len([r for r in pipeline_results if r['final_confidence']]):.3f}")

# Performance by true label
print(f"\nPerformance by True Label:")
for label in ['Neither', 'Cancer Only', 'Diabetes Only', 'Both']:
    label_results = [r for r in pipeline_results if r['true_label'] == label]
    if label_results:
        label_accuracy = sum(r['correct'] for r in label_results) / len(label_results)
        print(f"  {label:20s}: {label_accuracy:.1%} ({sum(r['correct'] for r in label_results)}/{len(label_results)})")

# Cases where classification changed
changed_cases = [r for r in pipeline_results if r['initial_classification'] != r['final_classification']]
print(f"\nCases where final decision differed from initial: {len(changed_cases)} ({len(changed_cases)/len(pipeline_results)*100:.1f}%)")

# Improvement analysis
initial_correct = [r for r in pipeline_results if r['initial_classification'] == r['true_label']]
final_correct = [r for r in pipeline_results if r['final_classification'] == r['true_label']]
print(f"\nAccuracy Comparison:")
print(f"  Initial Classification: {len(initial_correct)/len(pipeline_results):.1%} ({len(initial_correct)}/{len(pipeline_results)})")
print(f"  Final Decision: {len(final_correct)/len(pipeline_results):.1%} ({len(final_correct)}/{len(pipeline_results)})")
improvement = len(final_correct) - len(initial_correct)
print(f"  Improvement: {'+' if improvement >= 0 else ''}{improvement} cases")

# Confusion matrices
print(f"\n{'='*80}")
print("CONFUSION MATRICES")
print("="*80)

print("\nCancer Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y_true_cancer, y_pred_cancer),
    index=['True: No Cancer', 'True: Cancer'],
    columns=['Pred: No Cancer', 'Pred: Cancer']
))

print("\nDiabetes Confusion Matrix:")
print(pd.DataFrame(
    confusion_matrix(y_true_diabetes, y_pred_diabetes),
    index=['True: No Diabetes', 'True: Diabetes'],
    columns=['Pred: No Diabetes', 'Pred: Diabetes']
))

print("\nCombined Label Confusion Matrix:")
label_order = ['Neither', 'Cancer Only', 'Diabetes Only', 'Both']
cm_combined = confusion_matrix(y_true_combined, y_pred_combined, labels=label_order)
print(pd.DataFrame(
    cm_combined,
    index=[f'True: {l}' for l in label_order],
    columns=[f'Pred: {l}' for l in label_order]
))

# Save results to dataframe
results_df = pd.DataFrame(pipeline_results)
print(f"\n{'='*80}")
print(f"Results saved to dataframe: results_df")
print(f"Total test cases: {len(results_df)}")

COMPLETE PIPELINE TEST - THREE AGENT SYSTEM

Processing case 1/10...
Processing case 2/10...
Processing case 3/10...
Processing case 4/10...
Processing case 5/10...
Processing case 6/10...
Processing case 7/10...
Processing case 8/10...
Processing case 9/10...
Processing case 10/10...
FINAL EVALUATION STATISTICS

Overall Accuracy: 100.0% (10/10)

Cancer Detection Accuracy: 100.0%

Cancer Classification Report:
              precision    recall  f1-score   support

   No Cancer       1.00      1.00      1.00         6
      Cancer       1.00      1.00      1.00         4

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10


Diabetes Detection Accuracy: 100.0%

Diabetes Classification Report:
              precision    recall  f1-score   support

 No Diabetes       1.00      1.00      1.00         9
    Diabetes       1.00      1.00      1.00         1

    accuracy             